In [344]:
# libraries

import gurobipy as gp
from gurobipy import GRB
import numpy as np

import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import time
import random

from scipy import stats
import causalicp as icp
from sklearn.cluster import KMeans

import torch
from itertools import combinations

### Task 1: an invariant prediction

#### Vanilla mip

In [4]:
def vanilla_msetype(
    X_list, y_list, n_list=None, gamma1=1.0, rho=1.0, lam=1.0,
    s_max=None, beta_bound=100.0, S_start=None, B_start=None,
    mip_focus=2, mip_cuts=2, forbidden=None, time_limit=None,
    mip_gap=None, verbose=True
):
    E = len(X_list)
    if E == 0: raise ValueError("X_list must contain at least one environment.")

    X_list = [np.asarray(X, float) for X in X_list]
    y_list = [np.asarray(y, float).reshape(-1) for y in y_list]
    p = X_list[0].shape[1]

    for e, (X, y) in enumerate(zip(X_list, y_list)):
        if X.ndim != 2 or X.shape[1] != p or X.shape[0] != len(y):
            raise ValueError(f"Incompatible X/y dimensions in environment {e}.")

    n_list = np.array([len(y) for y in y_list], float) if n_list is None else np.asarray(n_list, float)
    if len(n_list) != E or np.any(n_list <= 0): raise ValueError("Invalid n_list.")

    w = n_list / n_list.sum()
    w_pair = np.outer(w, w)

    Sigma = np.array([X.T @ X / n for X, n in zip(X_list, n_list)])
    Xty = np.array([X.T @ y / n for X, y, n in zip(X_list, y_list, n_list)])
    y2 = np.array([y @ y / n for y, n in zip(y_list, n_list)])

    XtX_pool = sum(X.T @ X for X in X_list)
    Xty_pool = sum(X.T @ y for X, y in zip(X_list, y_list))

    model = gp.Model("finite_sample_invariance_indicator_miqp")
    if not verbose: model.Params.OutputFlag = 0
    if time_limit is not None: model.Params.TimeLimit = time_limit
    if mip_gap is not None: model.Params.MIPGap = mip_gap
    model.Params.MIPFocus, model.Params.Cuts = mip_focus, mip_cuts

    g = model.addVars(p, vtype=GRB.BINARY, name="g")
    beta = model.addVars(p, lb=-beta_bound, ub=beta_bound, name="beta")
    beta_e = model.addVars(E, p, lb=-beta_bound, ub=beta_bound, name="beta_e")

    if s_max is not None:
        model.addConstr(gp.quicksum(g[j] for j in range(p)) <= s_max, name="sparsity")

    if forbidden is not None:
        for S in forbidden:
            S = set(S)
            model.addConstr(
                gp.quicksum(1 - g[j] for j in S)
                + gp.quicksum(g[j] for j in range(p) if j not in S) >= 1
            )

    for j in range(p):
        model.addGenConstrIndicator(g[j], 0, beta[j] == 0, name=f"beta_support_{j}")
        for e in range(E):
            model.addGenConstrIndicator(g[j], 0, beta_e[e, j] == 0, name=f"beta_e_support_{e}_{j}")
            stationarity = gp.quicksum(Sigma[e, j, k] * beta_e[e, k] for k in range(p)) - Xty[e, j]
            model.addGenConstrIndicator(g[j], 1, stationarity == 0, name=f"ols_stationarity_{e}_{j}")

    if B_start is not None and S_start is None:
        raise ValueError("B_start requires S_start.")

    if S_start is not None:
        S0 = np.array(sorted(set(S_start)), int)
        if np.any(S0 < 0) or np.any(S0 >= p): raise ValueError("S_start contains invalid indices.")
        if s_max is not None and len(S0) > s_max: raise ValueError("S_start violates s_max.")

        if B_start is None:
            B_start = np.zeros((E + 1, p))
            if len(S0):
                B_start[0, S0] = np.linalg.solve(XtX_pool[np.ix_(S0, S0)], Xty_pool[S0])
                for e in range(E):
                    B_start[e + 1, S0] = np.linalg.solve(
                        Sigma[e][np.ix_(S0, S0)], Xty[e, S0]
                    )
        else:
            B_start = np.asarray(B_start, float)
            if B_start.shape != (E + 1, p):
                raise ValueError(f"B_start must have shape {(E + 1, p)}.")

        S0 = set(S0)
        for j in range(p):
            g[j].Start = float(j in S0)
            beta[j].Start = B_start[0, j]
        for e in range(E):
            for j in range(p):
                beta_e[e, j].Start = B_start[e + 1, j]

    obj = gp.QuadExpr()

    for e in range(E):
        obj += w[e] * y2[e]
        obj += gp.quicksum(-2 * w[e] * Xty[e, j] * beta[j] for j in range(p))
        obj += gp.quicksum(
            w[e] * Sigma[e, j, k] * beta[j] * beta[k]
            for j in range(p) for k in range(p) if Sigma[e, j, k] != 0
        )

    for e in range(E):
        for f in range(e + 1, E):
            wef = w_pair[e, f]

            obj += gamma1 * wef * gp.quicksum(
                (beta_e[e, j] - beta_e[f, j]) ** 2 for j in range(p)
            )

            var_diff = (
                y2[e] - y2[f]
                - gp.quicksum(Xty[e, j] * beta_e[e, j] for j in range(p))
                + gp.quicksum(Xty[f, j] * beta_e[f, j] for j in range(p))
            )
            obj += gamma1 * rho * wef * var_diff ** 2

    obj += lam * gp.quicksum(g[j] for j in range(p))
    model.setObjective(obj, GRB.MINIMIZE)
    model.optimize()

    if model.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        raise RuntimeError(f"Gurobi did not return a usable solution. Status = {model.Status}")
    if model.SolCount == 0: raise RuntimeError("Gurobi did not find a feasible solution.")

    g_sol = np.array([g[j].X > 0.5 for j in range(p)], int)
    B = np.zeros((E + 1, p))
    B[0] = [beta[j].X for j in range(p)]
    for e in range(E): B[e + 1] = [beta_e[e, j].X for j in range(p)]

    return model.ObjVal, B, g_sol

#### Coordinate descent

In [5]:
def precompute_stat(X_list, y_list, n_list=None):
    E = len(X_list)
    n_list = np.array([len(y) for y in y_list], float) if n_list is None else np.asarray(n_list, float)
    w = n_list / n_list.sum()
    y2 = [float(y @ y) for y in y_list]
    Xty = [X.T @ y for X, y in zip(X_list, y_list)]
    XtX = [X.T @ X for X in X_list]

    return {
        "E": E, "n_list": n_list, "w": w, "w_pair": np.outer(w, w),
        "y2_list": y2, "Xty_list": Xty, "XtX_list": XtX,
        "Xty_pool": np.sum(Xty, axis=0), "XtX_pool": np.sum(XtX, axis=0)
    }

In [6]:
def _add_feature_single(beta, S, Ginv, j, XtX, Xty, tol=1e-12):
    S, p = np.asarray(S, int), len(beta)
    if j in S: raise ValueError(f"j={j} is already in S.")

    if len(S) == 0:
        delta = XtX[j, j]
        if delta <= tol: raise np.linalg.LinAlgError(f"Cannot add j={j}: Schur complement too small.")
        S_new, Ginv_new = np.array([j]), np.array([[1 / delta]])
    else:
        c, d = XtX[S, j], XtX[j, j]
        u, delta = Ginv @ c, d - c @ Ginv @ c
        if delta <= tol: raise np.linalg.LinAlgError(f"Cannot add j={j}: Schur complement too small ({delta}).")
        k = len(S)
        Gtmp = np.empty((k + 1, k + 1))
        Gtmp[:k, :k] = Ginv + np.outer(u, u) / delta
        Gtmp[:k, k] = Gtmp[k, :k] = -u / delta
        Gtmp[k, k] = 1 / delta
        Stmp = np.r_[S, j]
        perm = np.argsort(Stmp)
        S_new, Ginv_new = Stmp[perm], Gtmp[np.ix_(perm, perm)]
        Ginv_new = (Ginv_new + Ginv_new.T) / 2

    beta_new = np.zeros(p)
    beta_new[S_new] = Ginv_new @ Xty[S_new]
    return beta_new, S_new, Ginv_new

In [7]:
def add_feature_to_B_ols(B, S, j, Ginv_pool, Ginv_list, stat, tol=1e-12):
    E = stat["E"]
    B_new = np.zeros_like(B, dtype=float)
    B_new[0], S_new, Ginv_pool_new = _add_feature_single(
        B[0], S, Ginv_pool, j, stat["XtX_pool"], stat["Xty_pool"], tol
    )
    Ginv_list_new = []
    for e in range(E):
        B_new[e + 1], S_e, G_e = _add_feature_single(
            B[e + 1], S, Ginv_list[e], j, stat["XtX_list"][e], stat["Xty_list"][e], tol
        )
        if not np.array_equal(S_e, S_new): raise RuntimeError("Support mismatch across environments.")
        Ginv_list_new.append(G_e)

    return B_new, S_new, Ginv_pool_new, Ginv_list_new

In [8]:
def _remove_feature_single(beta, S, Ginv, j, Xty, tol=1e-12):
    S, p = np.asarray(S, int), len(beta)
    if j not in S: raise ValueError(f"j={j} is not in S.")
    if len(S) == 1:
        return np.zeros(p), np.array([], int), np.zeros((0, 0))
    pos = np.where(S == j)[0][0]
    keep = np.arange(len(S)) != pos
    d = Ginv[pos, pos]
    if abs(d) <= tol: raise np.linalg.LinAlgError(f"Cannot remove j={j}: pivot too small.")
    col, row = Ginv[keep, pos], Ginv[pos, keep]
    Ginv_new = Ginv[np.ix_(keep, keep)] - np.outer(col, row) / d
    Ginv_new = (Ginv_new + Ginv_new.T) / 2
    S_new = S[keep]
    beta_new = np.zeros(p)
    beta_new[S_new] = Ginv_new @ Xty[S_new]
    return beta_new, S_new, Ginv_new

In [9]:
def remove_feature_from_B_ols(B, S, j, Ginv_pool, Ginv_list, stat, tol=1e-12):
    E = stat["E"]
    B_new = np.zeros_like(B, dtype=float)
    B_new[0], S_new, Ginv_pool_new = _remove_feature_single(
        B[0], S, Ginv_pool, j, stat["Xty_pool"], tol
    )
    Ginv_list_new = []
    for e in range(E):
        B_new[e + 1], S_e, G_e = _remove_feature_single(
            B[e + 1], S, Ginv_list[e], j, stat["Xty_list"][e], tol
        )
        if not np.array_equal(S_e, S_new): raise RuntimeError("Support mismatch across environments.")
        Ginv_list_new.append(G_e)
        
    return B_new, S_new, Ginv_pool_new, Ginv_list_new

In [10]:
def compute_Ginv_given_S(S, stat):
    S, E = np.asarray(S, int), stat["E"]
    if len(S) == 0:
        return np.zeros((0, 0)), [np.zeros((0, 0)) for _ in range(E)]
    Ginv_pool = np.linalg.inv(stat["XtX_pool"][np.ix_(S, S)])
    Ginv_list = [np.linalg.inv(stat["XtX_list"][e][np.ix_(S, S)]) for e in range(E)]
    return Ginv_pool, Ginv_list

In [11]:
def compute_B_given_S(S, stat):
    S = np.asarray(S, int)
    E, p = stat["E"], stat["XtX_pool"].shape[0]
    B = np.zeros((E + 1, p))
    if len(S) == 0: return B
    Ginv_pool, Ginv_list = compute_Ginv_given_S(S, stat)
    B[0, S] = Ginv_pool @ stat["Xty_pool"][S]
    for e in range(E): B[e + 1, S] = Ginv_list[e] @ stat["Xty_list"][e][S]
    return B

In [12]:
def compute_mse_obj_given_S(B, S, stat, gamma1=1.0, rho=1.0, lam=0.0, return_parts=False):
    E, n, w, w_pair = stat["E"], stat["n_list"], stat["w"], stat["w_pair"]
    y2, Xty, XtX = stat["y2_list"], stat["Xty_list"], stat["XtX_list"]
    beta = B[0]
    pred = sum(
        w[e] * (y2[e] - 2 * Xty[e] @ beta + beta @ XtX[e] @ beta) / n[e]
        for e in range(E)
    )
    resid = [(y2[e] - Xty[e] @ B[e + 1]) / n[e] for e in range(E)]
    coef_diff, var_diff = 0.0, 0.0
    for e in range(E):
        for f in range(e + 1, E):
            d = B[e + 1] - B[f + 1]
            coef_diff += w_pair[e, f] * (d @ d)
            var_diff += w_pair[e, f] * (resid[e] - resid[f]) ** 2
    sparsity = lam * len(S)
    obj = pred + gamma1 * (coef_diff + rho * var_diff) + sparsity
    if return_parts:
        return obj, {
            "prediction": pred,
            "coef_diff_raw": coef_diff,
            "coef_diff": gamma1 * coef_diff,
            "var_diff_raw": var_diff,
            "var_diff": gamma1 * rho * var_diff,
            "sparsity": sparsity,
            "total": obj
        }
    return obj

In [13]:
def cd_msetype(
    stat, lam, gamma1, rho, forbid=None, S_init=(), superset_init=(), tol=1e-10, max_iter=100
):
    p = stat["XtX_pool"].shape[0]
    forbid = set() if forbid is None else set(forbid)
    S = set(S_init) - forbid
    superset = (set(superset_init) - forbid) | S
    S0 = np.array(sorted(S), int)
    B = compute_B_given_S(S0, stat)
    Ginv_pool, Ginv_list = compute_Ginv_given_S(S0, stat)
    obj = compute_mse_obj_given_S(B, S0, stat, gamma1, rho, lam)

    for _ in range(max_iter):

        # Stage 1: converge within active set
        while True:
            improved = False
            for j in sorted(superset):
                S_curr = np.array(sorted(S), int)
                if j in S:
                    out = remove_feature_from_B_ols(B, S_curr, j, Ginv_pool, Ginv_list, stat, tol)
                else:
                    if j in forbid: continue
                    out = add_feature_to_B_ols(B, S_curr, j, Ginv_pool, Ginv_list, stat, tol)
                B_new, S_new, Gp_new, Ge_new = out
                obj_new = compute_mse_obj_given_S(B_new, S_new, stat, gamma1, rho, lam)
                if obj_new < obj - tol:
                    S, B, Ginv_pool, Ginv_list, obj = set(S_new), B_new, Gp_new, Ge_new, obj_new
                    improved = True
            if not improved: break

        # Stage 2: first improving global move
        new_feature = False
        for j in range(p):
            S_curr = np.array(sorted(S), int)
            if j in S:
                out = remove_feature_from_B_ols(B, S_curr, j, Ginv_pool, Ginv_list, stat, tol)
            else:
                if j in forbid: continue
                out = add_feature_to_B_ols(B, S_curr, j, Ginv_pool, Ginv_list, stat, tol)
            B_new, S_new, Gp_new, Ge_new = out
            obj_new = compute_mse_obj_given_S(B_new, S_new, stat, gamma1, rho, lam)
            if obj_new < obj - tol:
                S, B, Ginv_pool, Ginv_list, obj = set(S_new), B_new, Gp_new, Ge_new, obj_new
                if j not in superset and j in S:
                    superset.add(j)
                    new_feature = True
                break
        if not new_feature: break

    S = sorted(map(int, S))
    return S, B, compute_mse_obj_given_S(B, S, stat, gamma1, rho, lam)

### Task 2: the intersection

#### Vanilla mip

In [355]:
def vanilla_testtype(
    X_list, y_list, k0, n_list=None, gamma=(1,1,1,1),
    s_max=None, beta_bound=100., S_start=None, B_start=None,
    mip_focus=2, mip_cuts=2, forbidden=None, time_limit=None,
    mip_gap=None, verbose=True
):
    gamma = np.asarray(gamma, float)
    if gamma.shape != (4,) or np.any(gamma < 0):
        raise ValueError("gamma must contain four nonnegative values.")
    gamma1, gamma2, gamma3, gamma4 = gamma

    E = len(X_list)
    if E == 0: raise ValueError("X_list must contain at least one environment.")

    X_list = [np.asarray(X, float) for X in X_list]
    y_list = [np.asarray(y, float).reshape(-1) for y in y_list]
    p = X_list[0].shape[1]
    if not 0 <= k0 < p: raise ValueError("Invalid k0.")

    for e, (X, y) in enumerate(zip(X_list, y_list)):
        if X.ndim != 2 or X.shape != (len(y), p):
            raise ValueError(f"Incompatible X/y dimensions in environment {e}.")

    n_list = np.array([len(y) for y in y_list], float) if n_list is None else np.asarray(n_list, float)
    if len(n_list) != E or np.any(n_list <= 0): raise ValueError("Invalid n_list.")

    w = n_list / n_list.sum()
    wp = np.outer(w, w)

    Sigma = np.array([X.T @ X/n for X, n in zip(X_list, n_list)])
    Xty = np.array([X.T @ y/n for X, y, n in zip(X_list, y_list, n_list)])
    y2 = np.array([y @ y/n for y, n in zip(y_list, n_list)])
    xm = np.array([X.sum(0)/n for X, n in zip(X_list, n_list)])
    ym = np.array([y.sum()/n for y, n in zip(y_list, n_list)])

    XtX_pool = sum(X.T @ X for X in X_list)
    Xty_pool = sum(X.T @ y for X, y in zip(X_list, y_list))

    model = gp.Model("mip_test_four_term")
    if not verbose: model.Params.OutputFlag = 0
    if time_limit is not None: model.Params.TimeLimit = time_limit
    if mip_gap is not None: model.Params.MIPGap = mip_gap
    model.Params.MIPFocus, model.Params.Cuts = mip_focus, mip_cuts

    g = model.addVars(p, vtype=GRB.BINARY, name="g")
    beta = model.addVars(p, lb=-beta_bound, ub=beta_bound, name="beta")
    beta_e = model.addVars(E, p, lb=-beta_bound, ub=beta_bound, name="beta_e")
    t1, t2, t3, t4 = [model.addVar(lb=0., name=f"t{i}") for i in range(1,5)]

    model.addConstr(g[k0] == 0, name=f"exclude_{k0}")
    if s_max is not None:
        model.addConstr(gp.quicksum(g[j] for j in range(p)) <= s_max, name="sparsity")

    if forbidden is not None:
        for S in forbidden:
            S = set(S)
            model.addConstr(
                gp.quicksum(1-g[j] for j in S) +
                gp.quicksum(g[j] for j in range(p) if j not in S) >= 1
            )

    # Support + environment/pooled OLS
    for j in range(p):
        model.addGenConstrIndicator(g[j], 0, beta[j] == 0, name=f"beta_support_{j}")

        for e in range(E):
            model.addGenConstrIndicator(g[j], 0, beta_e[e,j] == 0, name=f"beta_e_support_{e}_{j}")
            stat_e = gp.quicksum(Sigma[e,j,k]*beta_e[e,k] for k in range(p)) - Xty[e,j]
            model.addGenConstrIndicator(g[j], 1, stat_e == 0, name=f"env_ols_{e}_{j}")

        stat_pool = gp.quicksum(
            w[e] * (gp.quicksum(Sigma[e,j,k]*beta[k] for k in range(p)) - Xty[e,j])
            for e in range(E)
        )
        model.addGenConstrIndicator(g[j], 1, stat_pool == 0, name=f"pooled_ols_{j}")

    # (13a): coefficient mismatch
    q1 = gp.quicksum(
        wp[e,f] * gp.quicksum((beta_e[e,j]-beta_e[f,j])**2 for j in range(p))
        for e in range(E) for f in range(e+1,E)
    )
    model.addQConstr(q1 <= t1, name="term1_coef")

    # (13b): residual variance mismatch
    q2 = gp.QuadExpr()
    for e in range(E):
        for f in range(e+1,E):
            d = (
                y2[e] - y2[f]
                - gp.quicksum(Xty[e,j]*beta_e[e,j] for j in range(p))
                + gp.quicksum(Xty[f,j]*beta_e[f,j] for j in range(p))
            )
            q2 += wp[e,f] * d**2
    model.addQConstr(q2 <= t2, name="term2_var")

    # (13c): residual mean mismatch
    rmean = [
        ym[e] - gp.quicksum(xm[e,j]*beta[j] for j in range(p))
        for e in range(E)
    ]
    q3 = gp.quicksum(
        wp[e,f] * (rmean[e]-rmean[f])**2
        for e in range(E) for f in range(e+1,E)
    )
    model.addQConstr(q3 <= t3, name="term3_mean")

    # (13d): residual normal-equation mismatch for active features
    q4_aux = model.addVars(p, lb=0., name="q4_aux")

    for k in range(p):
        rcorr = [
            Xty[e,k] - gp.quicksum(Sigma[e,k,j]*beta[j] for j in range(p))
            for e in range(E)
        ]
        q4 = gp.quicksum(
            wp[e,f] * (rcorr[e]-rcorr[f])**2
            for e in range(E) for f in range(e+1,E)
        )
        model.addQConstr(q4 <= q4_aux[k], name=f"term4_quad_{k}")
        model.addGenConstrIndicator(g[k], 1, q4_aux[k] <= t4, name=f"term4_active_{k}")

    # Warm start
    if B_start is not None and S_start is None:
        raise ValueError("B_start requires S_start.")

    if S_start is not None:
        S0 = np.array(sorted(set(S_start)), int)
        if np.any(S0 < 0) or np.any(S0 >= p): raise ValueError("Invalid S_start.")
        if k0 in S0: raise ValueError("S_start cannot contain k0.")
        if s_max is not None and len(S0) > s_max: raise ValueError("S_start violates s_max.")

        if B_start is None:
            B_start = np.zeros((E+1,p))
            if len(S0):
                B_start[0,S0] = np.linalg.solve(
                    XtX_pool[np.ix_(S0,S0)], Xty_pool[S0]
                )
                for e in range(E):
                    B_start[e+1,S0] = np.linalg.solve(
                        Sigma[e][np.ix_(S0,S0)], Xty[e,S0]
                    )
        else:
            B_start = np.asarray(B_start, float)
            if B_start.shape != (E+1,p):
                raise ValueError(f"B_start must have shape {(E+1,p)}.")

        S0 = set(S0)
        for j in range(p):
            g[j].Start = float(j in S0)
            beta[j].Start = B_start[0,j]
        for e in range(E):
            for j in range(p):
                beta_e[e,j].Start = B_start[e+1,j]

    model.setObjective(gamma1*t1 + gamma2*t2 + gamma3*t3 + gamma4*t4, GRB.MINIMIZE)
    model.optimize()

    if model.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        raise RuntimeError(f"Gurobi returned status {model.Status}.")
    if model.SolCount == 0: raise RuntimeError("No feasible solution found.")

    g_sol = np.array([g[j].X > .5 for j in range(p)], int)
    B = np.zeros((E+1,p))
    B[0] = [beta[j].X for j in range(p)]
    for e in range(E): B[e+1] = [beta_e[e,j].X for j in range(p)]

    return model.ObjVal, B, g_sol

#### Coordinate descent

In [2246]:
def precompute_stat_test(X_list, y_list, n_list=None):
    E = len(X_list)
    n = np.array([len(y) for y in y_list], float) if n_list is None else np.asarray(n_list, float)
    w = n / n.sum()

    y2 = [float(y @ y) for y in y_list]
    Xty = [X.T @ y for X, y in zip(X_list, y_list)]
    XtX = [X.T @ X for X in X_list]

    return {
        "E": E, "n_list": n, "w": w, "w_pair": np.outer(w, w),
        "y2_list": y2, "Xty_list": Xty, "XtX_list": XtX,
        "xmean_list": [X.mean(0) for X in X_list],
        "ymean_list": [float(y.mean()) for y in y_list],
        "Xty_pool": np.sum(Xty, axis=0),
        "XtX_pool": np.sum(XtX, axis=0)
    }

In [2247]:
def _add_feature_single(beta, S, Ginv, j, XtX, Xty, tol=1e-12):
    S, p = np.asarray(S, int), len(beta)
    if j in S: raise ValueError(f"j={j} already in S.")

    if len(S) == 0:
        delta = XtX[j, j]
        if delta <= tol: raise np.linalg.LinAlgError(f"Cannot add j={j}.")
        S_new, G_new = np.array([j]), np.array([[1 / delta]])
    else:
        c, d = XtX[S, j], XtX[j, j]
        u, delta = Ginv @ c, d - c @ Ginv @ c
        if delta <= tol: raise np.linalg.LinAlgError(f"Cannot add j={j}.")

        k = len(S)
        G = np.empty((k + 1, k + 1))
        G[:k, :k] = Ginv + np.outer(u, u) / delta
        G[:k, k] = G[k, :k] = -u / delta
        G[k, k] = 1 / delta

        Stmp = np.r_[S, j]
        perm = np.argsort(Stmp)
        S_new, G_new = Stmp[perm], G[np.ix_(perm, perm)]
        G_new = (G_new + G_new.T) / 2

    beta_new = np.zeros(p)
    beta_new[S_new] = G_new @ Xty[S_new]
    return beta_new, S_new, G_new

In [2248]:
def add_feature_to_B_ols(B, S, j, Gp, Ge, stat, tol=1e-12):
    E = stat["E"]
    B_new = np.zeros_like(B, float)

    B_new[0], S_new, Gp_new = _add_feature_single(
        B[0], S, Gp, j, stat["XtX_pool"], stat["Xty_pool"], tol
    )

    Ge_new = []
    for e in range(E):
        B_new[e+1], S_e, G_e = _add_feature_single(
            B[e+1], S, Ge[e], j, stat["XtX_list"][e], stat["Xty_list"][e], tol
        )
        if not np.array_equal(S_e, S_new): raise RuntimeError("Support mismatch.")
        Ge_new.append(G_e)

    return B_new, S_new, Gp_new, Ge_new

In [2249]:
def _remove_feature_single(beta, S, Ginv, j, Xty, tol=1e-12):
    S, p = np.asarray(S, int), len(beta)
    if j not in S: raise ValueError(f"j={j} not in S.")
    if len(S) == 1: return np.zeros(p), np.array([], int), np.zeros((0, 0))

    pos = np.where(S == j)[0][0]
    keep = np.arange(len(S)) != pos
    d = Ginv[pos, pos]
    if abs(d) <= tol: raise np.linalg.LinAlgError(f"Cannot remove j={j}.")

    G_new = Ginv[np.ix_(keep, keep)] - np.outer(Ginv[keep, pos], Ginv[pos, keep]) / d
    G_new = (G_new + G_new.T) / 2
    S_new = S[keep]

    beta_new = np.zeros(p)
    beta_new[S_new] = G_new @ Xty[S_new]
    return beta_new, S_new, G_new

In [2250]:
def remove_feature_from_B_ols(B, S, j, Gp, Ge, stat, tol=1e-12):
    E = stat["E"]
    B_new = np.zeros_like(B, float)

    B_new[0], S_new, Gp_new = _remove_feature_single(
        B[0], S, Gp, j, stat["Xty_pool"], tol
    )

    Ge_new = []
    for e in range(E):
        B_new[e+1], S_e, G_e = _remove_feature_single(
            B[e+1], S, Ge[e], j, stat["Xty_list"][e], tol
        )
        if not np.array_equal(S_e, S_new): raise RuntimeError("Support mismatch.")
        Ge_new.append(G_e)

    return B_new, S_new, Gp_new, Ge_new

In [2251]:
def compute_Ginv_given_S(S, stat):
    S, E = np.asarray(S, int), stat["E"]
    if len(S) == 0:
        return np.zeros((0, 0)), [np.zeros((0, 0)) for _ in range(E)]

    Gp = np.linalg.inv(stat["XtX_pool"][np.ix_(S, S)])
    Ge = [
        np.linalg.inv(stat["XtX_list"][e][np.ix_(S, S)])
        for e in range(E)
    ]
    return Gp, Ge

In [2252]:
def compute_B_given_S(S, stat):
    S = np.asarray(S, int)
    E, p = stat["E"], stat["XtX_pool"].shape[0]
    B = np.zeros((E + 1, p))
    if len(S) == 0: return B

    Gp, Ge = compute_Ginv_given_S(S, stat)
    B[0, S] = Gp @ stat["Xty_pool"][S]
    for e in range(E):
        B[e+1, S] = Ge[e] @ stat["Xty_list"][e][S]

    return B

In [2253]:
def compute_test_obj_given_S(B, S, stat, gamma, return_parts=False):
    gamma = np.asarray(gamma, float)
    if gamma.shape != (4,) or np.any(gamma < 0):
        raise ValueError("gamma must contain four nonnegative values.")
    gamma1, gamma2, gamma3, gamma4 = gamma

    E, n, wp = stat["E"], stat["n_list"], stat["w_pair"]
    y2, Xty, XtX = stat["y2_list"], stat["Xty_list"], stat["XtX_list"]
    xm, ym, beta = stat["xmean_list"], stat["ymean_list"], B[0]
    S = set(map(int, S))

    t1 = sum(
        wp[e,f] * np.sum((B[e+1]-B[f+1])**2)
        for e in range(E) for f in range(e+1,E)
    )

    rvar = np.array([(y2[e]-Xty[e]@B[e+1])/n[e] for e in range(E)])
    t2 = sum(
        wp[e,f] * (rvar[e]-rvar[f])**2
        for e in range(E) for f in range(e+1,E)
    )

    rmean = np.array([ym[e]-xm[e]@beta for e in range(E)])
    t3 = sum(
        wp[e,f] * (rmean[e]-rmean[f])**2
        for e in range(E) for f in range(e+1,E)
    )

    rcorr = np.array([(Xty[e]-XtX[e]@beta)/n[e] for e in range(E)])
    t4_each = sum(
        wp[e,f] * (rcorr[e]-rcorr[f])**2
        for e in range(E) for f in range(e+1,E)
    )
    active = list(S)
    t4 = np.max(t4_each[active]) if active else 0.

    obj = gamma1*t1 + gamma2*t2 + gamma3*t3 + gamma4*t4

    if return_parts:
        return obj, {
            "t1": t1, "t2": t2, "t3": t3, "t4": t4,
            "term1": gamma1*t1, "term2": gamma2*t2,
            "term3": gamma3*t3, "term4": gamma4*t4,
            "total": obj
        }
    return obj

In [2254]:
def cd_testtype(
    stat, k0, gamma = (1,1,1,1),
    forbid=None, S_init=(), superset_init=(), tol=1e-10, max_iter=100
):
    p = stat["XtX_pool"].shape[0]
    if not 0 <= k0 < p: raise ValueError("Invalid k0.")

    forbid = (set() if forbid is None else set(forbid)) | {k0}
    S = set(S_init) - forbid
    superset = (set(superset_init) - forbid) | S

    S0 = np.array(sorted(S), int)
    B = compute_B_given_S(S0, stat)
    Gp, Ge = compute_Ginv_given_S(S0, stat)
    obj = compute_test_obj_given_S(B, S0, stat, gamma)

    for _ in range(max_iter):

        # Stage 1
        while True:
            improved = False
            for j in sorted(superset):
                S0 = np.array(sorted(S), int)
                out = (
                    remove_feature_from_B_ols(B, S0, j, Gp, Ge, stat, tol)
                    if j in S else
                    add_feature_to_B_ols(B, S0, j, Gp, Ge, stat, tol)
                )
                Bn, Sn, Gpn, Gen = out
                objn = compute_test_obj_given_S(
                    Bn, Sn, stat, gamma
                )
                if objn < obj - tol:
                    S, B, Gp, Ge, obj = set(Sn), Bn, Gpn, Gen, objn
                    improved = True

            if not improved: break

        # Stage 2
        new_feature = False

        for j in range(p):
            if j in forbid: continue
            S0 = np.array(sorted(S), int)
            out = (
                remove_feature_from_B_ols(B, S0, j, Gp, Ge, stat, tol)
                if j in S else
                add_feature_to_B_ols(B, S0, j, Gp, Ge, stat, tol)
            )
            Bn, Sn, Gpn, Gen = out
            objn = compute_test_obj_given_S(
                Bn, Sn, stat, gamma
            )
            if objn < obj - tol:
                S, B, Gp, Ge, obj = set(Sn), Bn, Gpn, Gen, objn

                if j not in superset and j in S:
                    superset.add(j)
                    new_feature = True
                break

        if not new_feature: break

    S = sorted(map(int, S))
    return S, B, compute_test_obj_given_S(
        B, S, stat, gamma
    )

#### Bootstrap split

In [1440]:
def train_test_split_envs(X_list, y_list, test_size=.5, seed=0, shuffle=True):
    rng = np.random.default_rng(seed)
    X_train, y_train, X_test, y_test = [], [], [], []

    for X, y in zip(X_list, y_list):
        n = len(y)
        idx = rng.permutation(n) if shuffle else np.arange(n)
        n_test = int(np.floor(test_size*n))
        te, tr = idx[:n_test], idx[n_test:]

        X_train.append(X[tr]); y_train.append(y[tr])
        X_test.append(X[te]); y_test.append(y[te])

    n_train_list = [len(y) for y in y_train]
    n_test_list = [len(y) for y in y_test]

    return X_train, y_train, n_train_list, X_test, y_test, n_test_list

#### Bootstrap

In [365]:
def _pinv(A, rcond=1e-10):
    return np.linalg.pinv(A, rcond=rcond)


def _ols(X, y, rcond=1e-10):
    return np.zeros(0) if X.shape[1] == 0 else _pinv(X.T @ X, rcond) @ (X.T @ y)


def _pooled_ols(X_list, y_list, S, rcond=1e-10):
    S = np.asarray(S, int)
    if len(S) == 0: return np.zeros(0)
    XtX = sum(X[:, S].T @ X[:, S] for X in X_list)
    Xty = sum(X[:, S].T @ y for X, y in zip(X_list, y_list))
    return _pinv(XtX, rcond) @ Xty

In [366]:
def _bootstrap_cache(X_list, y_list, n_list, S, rcond=1e-10):
    S, N = np.asarray(S, int), int(np.sum(n_list))
    d = len(S)
    beta_pool = _pooled_ols(X_list, y_list, S, rcond)
    Sigma_pool, env = np.zeros((d, d)), []

    for X, y, ne in zip(X_list, y_list, n_list):
        XS = X[:, S] if d else np.zeros((ne, 0))
        beta_e = _ols(XS, y, rcond)
        re, rp = y - XS @ beta_e, y - XS @ beta_pool
        XtX = XS.T @ XS

        if d:
            Sigma_pool += XtX
            Sinv, mu, C, q = _pinv(XtX/ne, rcond), XS.mean(0), XtX/ne, XS.T @ rp/ne
        else:
            Sinv, mu, q, C = np.zeros((0,0)), np.zeros(0), np.zeros(0), np.zeros((0,0))

        env.append({
            "XS": XS, "re": re, "rp": rp, "Sinv": Sinv,
            "mu": mu, "C": C, "q": q,
            "m": float(rp.mean()), "v": float(np.mean(re**2))
        })

    Spinv = _pinv(Sigma_pool/N, rcond) if d else np.zeros((0,0))
    return {"S": S, "d": d, "Spinv": Spinv, "env": env}

In [472]:
def bootstrap_testtype(
    X_list, y_list, n_list, S_list, gamma=(1,1,1,1),
    B=300, random_state=None, rcond=1e-10, raw_scale=True
):
    gamma = np.asarray(gamma, float)
    if gamma.shape != (4,) or np.any(gamma < 0):
        raise ValueError("gamma must contain four nonnegative values.")

    X_list = [np.asarray(X, float) for X in X_list]
    y_list = [np.asarray(y, float).reshape(-1) for y in y_list]
    n = np.asarray(n_list, int)
    E, N = len(X_list), int(n.sum())
    w, pairs = n/N, list(combinations(range(E), 2))
    wp = np.outer(w, w)
    rng = np.random.default_rng(random_state)

    S_list = list(dict.fromkeys(tuple(sorted(map(int, S))) for S in S_list))
    if not S_list: raise ValueError("S_list cannot be empty.")
    cache = [_bootstrap_cache(X_list, y_list, n, S, rcond) for S in S_list]

    boot = np.zeros(B)

    for b in range(B):
        xi = [rng.normal(size=n[e]) for e in range(E)]
        best = -np.inf

        for c in cache:
            d, env = c["d"], c["env"]

            if d:
                score = sum(env[e]["XS"].T @ (xi[e]*env[e]["rp"]) for e in range(E)) / np.sqrt(N)
                Bstar = c["Spinv"] @ score
            else:
                Bstar = np.zeros(0)

            L, V, O, K = [], [], [], []

            for e in range(E):
                ne, sc, z, ce = n[e], np.sqrt(N/n[e]), xi[e], env[e]
                XS, re, rp = ce["XS"], ce["re"], ce["rp"]

                if d:
                    G = XS.T @ (z*re) / np.sqrt(ne)
                    Le = sc * ce["Sinv"] @ G
                    Xr = XS * rp[:, None]
                    Q = (z[:, None] * (Xr-ce["q"][None,:])).sum(0) / np.sqrt(ne)
                    Ke = sc*Q - ce["C"] @ Bstar
                else:
                    Le = Ke = np.zeros(0)

                H = np.sum(z*(re**2-ce["v"])) / np.sqrt(ne)
                M = np.sum(z*(rp-ce["m"])) / np.sqrt(ne)

                L.append(Le)
                V.append(sc*H)
                O.append(sc*M - ce["mu"] @ Bstar)
                K.append(Ke)

            t1 = t2 = t3 = 0.
            t4_each = np.zeros(d)

            for e, f in pairs:
                dL, dV, dO, dK = L[e]-L[f], V[e]-V[f], O[e]-O[f], K[e]-K[f]
                t1 += wp[e,f] * (dL @ dL)
                t2 += wp[e,f] * dV**2
                t3 += wp[e,f] * dO**2
                if d: t4_each += wp[e,f] * dK**2

            t4 = np.max(t4_each) if d else 0.
            best = max(best, gamma @ np.array([t1, t2, t3, t4]))

        boot[b] = best

    return boot/N if raw_scale else boot

In [368]:
def bootstrap_env_data(X_list, y_list, n_list, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    Xb, yb = [], []

    for X, y, ne in zip(X_list, y_list, n_list):
        idx = rng.integers(0, int(ne), int(ne))
        Xb.append(X[idx])
        yb.append(y[idx])

    return Xb, yb, np.asarray(n_list, int)

In [369]:
def _solve_testtype(
    X_list, y_list, n_list, k0, gamma, solver,
    superset, stat=None, solver_kwargs=None
):
    solver = solver.lower()
    kw = {} if solver_kwargs is None else dict(solver_kwargs)
    S0 = sorted(set(superset)-{k0})

    if solver == "cd":
        if stat is None: stat = precompute_stat_test(X_list, y_list, n_list)
        return cd_testtype(stat, k0, gamma=gamma, S_init=S0, superset_init=S0, **kw)

    if solver == "mip":
        obj, B, g = vanilla_testtype(
            X_list, y_list, k0, n_list=n_list, gamma=gamma, S_start=S0, **kw
        )
        return np.flatnonzero(g).tolist(), B, obj

    raise ValueError("solver must be 'cd' or 'mip'.")

In [370]:
def build_supercollection_test(
    X_list, y_list, n_list, k0, superset,
    gamma=(1,1,1,1), M_bar=5, solver="cd",
    random_state=None, solver_kwargs=None
):
    rng = np.random.default_rng(random_state)
    S_list = []

    for _ in range(M_bar):
        Xb, yb, nb = bootstrap_env_data(X_list, y_list, n_list, rng)
        S, _, _ = _solve_testtype(
            Xb, yb, nb, k0, gamma, solver, superset,
            solver_kwargs=solver_kwargs
        )
        S_list.append(tuple(sorted(S)))

    return [list(S) for S in dict.fromkeys(S_list)]

In [371]:
def mip_test_multiple(
    X_D1_list, y_D1_list, n_D1_list,
    X_D2_list, y_D2_list, n_D2_list,
    superset, gamma=(1,1,1,1), alpha=.05,
    M_bar=5, B=300, solver="cd",
    random_state=None, solver_kwargs=None,
    verbose=1, return_details=False
):
    p = X_D1_list[0].shape[1]
    rng = np.random.default_rng(random_state)

    collections = {
        k0: build_supercollection_test(
            X_D1_list, y_D1_list, n_D1_list, k0, superset,
            gamma=gamma, M_bar=M_bar, solver=solver,
            random_state=int(rng.integers(2**32)),
            solver_kwargs=solver_kwargs
        )
        for k0 in range(p)
    }

    stat_D2 = (
        precompute_stat_test(X_D2_list, y_D2_list, n_D2_list)
        if solver.lower() == "cd" else None
    )

    selected, details = [], {}

    for k0 in range(p):
        boot = bootstrap_testtype(
            X_D2_list, y_D2_list, n_D2_list, collections[k0],
            gamma=gamma, B=B, random_state=int(rng.integers(2**32))
        )
        thres = np.quantile(boot, 1-alpha/p).item()

        S, _, T = _solve_testtype(
            X_D2_list, y_D2_list, n_D2_list, k0,
            gamma, solver, superset, stat_D2, solver_kwargs
        )

        if T > thres: selected.append(k0)
        details[k0] = {"T": T, "threshold": thres, "S": S, "S_collection": collections[k0]}

        if verbose:
            print(f"k={k0}: T={T:.6g}, threshold={thres:.6g}, S={S}, M={len(collections[k0])}")

    return (selected, details) if return_details else selected

### Extension: confounding

In [938]:
def vanilla_msetype_additive(
    X_list, y_list, n_list=None, gamma=1., lam=1.,
    w=None, w_pair=None, s_max=None, beta_bound=100.,
    S_start=None, forbidden=None, mip_focus=2, mip_cuts=2,
    time_limit=None, mip_gap=None, verbose=True
):
    E = len(X_list)
    if E == 0: raise ValueError("X_list cannot be empty.")

    X_list = [np.asarray(X, float) for X in X_list]
    y_list = [np.asarray(y, float).reshape(-1) for y in y_list]
    p = X_list[0].shape[1]
    if any(X.shape != (len(y), p) for X, y in zip(X_list, y_list)):
        raise ValueError("Incompatible X/y dimensions.")

    n = np.array([len(y) for y in y_list], float) if n_list is None else np.asarray(n_list, float)
    if len(n) != E or np.any(n <= 0): raise ValueError("Invalid n_list.")

    w = n/n.sum() if w is None else np.asarray(w, float)
    wp = np.outer(w, w) if w_pair is None else np.asarray(w_pair, float)
    if w.shape != (E,) or wp.shape != (E, E): raise ValueError("Invalid weights.")

    Sigma = np.array([X.T@X/ne for X, ne in zip(X_list, n)])
    Xty = np.array([X.T@y/ne for X, y, ne in zip(X_list, y_list, n)])
    y2 = np.array([y@y/ne for y, ne in zip(y_list, n)])

    model = gp.Model("additive_invariance_miqp")
    if not verbose: model.Params.OutputFlag = 0
    if time_limit is not None: model.Params.TimeLimit = time_limit
    if mip_gap is not None: model.Params.MIPGap = mip_gap
    model.Params.MIPFocus, model.Params.Cuts = mip_focus, mip_cuts

    g = model.addVars(p, vtype=GRB.BINARY, name="g")
    beta = model.addVars(p, lb=-beta_bound, ub=beta_bound, name="beta")

    if s_max is not None:
        model.addConstr(gp.quicksum(g[j] for j in range(p)) <= s_max)

    if forbidden is not None:
        for S in forbidden:
            S = set(S)
            model.addConstr(
                gp.quicksum(1-g[j] for j in S)
                + gp.quicksum(g[j] for j in range(p) if j not in S) >= 1
            )

    for j in range(p):
        model.addGenConstrIndicator(g[j], 0, beta[j] == 0.)

    if S_start is not None:
        S0 = set(S_start)
        for j in range(p): g[j].Start = float(j in S0)

    obj = gp.QuadExpr()

    # weighted prediction error
    for e in range(E):
        obj += w[e] * (
            y2[e]
            - 2*gp.quicksum(Xty[e,j]*beta[j] for j in range(p))
            + gp.quicksum(
                Sigma[e,j,k]*beta[j]*beta[k]
                for j in range(p) for k in range(p)
                if Sigma[e,j,k] != 0
            )
        )

    # pairwise residual-moment discrepancy
    for e in range(E):
        for f in range(e+1, E):
            for j in range(p):
                d = (
                    Xty[e,j] - Xty[f,j]
                    + gp.quicksum(
                        (Sigma[f,j,k]-Sigma[e,j,k])*beta[k]
                        for k in range(p)
                        if Sigma[f,j,k] != Sigma[e,j,k]
                    )
                )
                obj += gamma * wp[e,f] * d*d

    obj += lam * gp.quicksum(g[j] for j in range(p))
    model.setObjective(obj, GRB.MINIMIZE)
    model.optimize()

    if model.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL] or model.SolCount == 0:
        raise RuntimeError(f"No usable solution. Status={model.Status}")

    beta_sol = np.array([beta[j].X for j in range(p)])
    g_sol = np.array([g[j].X > .5 for j in range(p)], int)

    return model.ObjVal, beta_sol, g_sol

### Extension: nonliner

#### Coordinate descent

In [823]:
from sklearn.preprocessing import SplineTransformer

def spline_expand(X_list, degree=3, n_knots=3, knots="quantile", standardize=True):
    X_list = [np.asarray(X, float) for X in X_list]
    X_pool, E, p = np.vstack(X_list), len(X_list), X_list[0].shape[1]
    Z_blocks, groups, basis_info, start = [[] for _ in range(E)], [], [], 0

    for j in range(p):
        sp = SplineTransformer(
            degree=degree, n_knots=n_knots, knots=knots, include_bias=False
        ).fit(X_pool[:, [j]])

        Zj = [sp.transform(X[:, [j]]) for X in X_list]
        Rj = Zj[0].shape[1]

        scale = np.ones(Rj)
        if standardize:
            scale = np.sqrt(np.mean(np.vstack(Zj)**2, axis=0))
            scale[scale < 1e-10] = 1.
            Zj = [Z / scale for Z in Zj]

        groups.append(np.arange(start, start + Rj))
        for e in range(E): Z_blocks[e].append(Zj[e])

        basis_info.append({"transformer": sp, "scale": scale, "Rj": Rj})
        start += Rj

    return [np.hstack(Z) for Z in Z_blocks], groups, basis_info

In [824]:
def fourier_expand(X_list, K=1):
    X_list = [np.asarray(X, float) for X in X_list]
    E, p = len(X_list), X_list[0].shape[1]
    Z_blocks, groups, start = [[] for _ in range(E)], [], 0

    for j in range(p):
        for e in range(E):
            x = X_list[e][:, j]
            Z_blocks[e].append(np.column_stack([
                f(k*x) for k in range(1, K+1) for f in (np.sin, np.cos)
            ]))
        Rj = 2*K
        groups.append(np.arange(start, start + Rj))
        start += Rj

    return [np.hstack(Z) for Z in Z_blocks], groups

In [825]:
def precompute_stat(X_list, y_list, n_list=None):
    X_list = [np.asarray(X, float) for X in X_list]
    y_list = [np.asarray(y, float).reshape(-1) for y in y_list]
    E = len(X_list)
    n = np.array([len(y) for y in y_list], float) if n_list is None else np.asarray(n_list, float)
    w = n / n.sum()
    y2 = [float(y @ y) for y in y_list]
    Xty = [X.T @ y for X, y in zip(X_list, y_list)]
    XtX = [X.T @ X for X in X_list]

    return {
        "E": E, 
        "n_list": n, 
        "w": w, 
        "w_pair": np.outer(w, w),
        "y2_list": y2, 
        "Xty_list": Xty, 
        "XtX_list": XtX,
        "Xty_pool": np.sum(Xty, axis=0), 
        "XtX_pool": np.sum(XtX, axis=0)
    }

In [741]:
def group_cols(S, groups):
    S = sorted(set(map(int, S)))
    if len(S) == 0:
        return np.array([], int)
    return np.sort(np.concatenate([groups[j] for j in S]))

In [826]:
def compute_B_given_groups(S, stat, groups, rcond=1e-10):
    A, E, R = group_cols(S, groups), stat["E"], stat["XtX_pool"].shape[0]
    B = np.zeros((E + 1, R))
    if not len(A): return B

    B[0, A] = np.linalg.pinv(
        stat["XtX_pool"][np.ix_(A, A)], rcond=rcond
    ) @ stat["Xty_pool"][A]

    for e in range(E):
        B[e + 1, A] = np.linalg.pinv(
            stat["XtX_list"][e][np.ix_(A, A)], rcond=rcond
        ) @ stat["Xty_list"][e][A]

    return B

In [827]:
def compute_mse_obj_given_groups(
    B, S, stat, gamma1=1., rho=1., lam=0., return_parts=False
):
    E, n, w, wp = stat["E"], stat["n_list"], stat["w"], stat["w_pair"]
    y2, Xty, XtX = stat["y2_list"], stat["Xty_list"], stat["XtX_list"]
    beta = B[0]

    pred = sum(
        w[e] * (y2[e] - 2*Xty[e]@beta + beta@XtX[e]@beta) / n[e]
        for e in range(E)
    )

    resid = [
        (y2[e] - Xty[e] @ B[e + 1]) / n[e]
        for e in range(E)
    ]

    coef_diff = var_diff = 0.
    for e in range(E):
        for f in range(e + 1, E):
            d = B[e + 1] - B[f + 1]
            coef_diff += wp[e, f] * d @ d
            var_diff += wp[e, f] * (resid[e] - resid[f])**2

    sparsity = lam * len(S)
    obj = pred + gamma1*coef_diff + gamma1*rho*var_diff + sparsity

    if not return_parts: return obj
    return obj, {
        "prediction": pred,
        "coef_diff_raw": coef_diff,
        "coef_diff": gamma1*coef_diff,
        "var_diff_raw": var_diff,
        "var_diff": gamma1*rho*var_diff,
        "sparsity": sparsity,
        "total": obj
    }

In [828]:
def cd_msetype_grouped(
    stat, groups, lam, gamma1, rho,
    forbid=None, S_init=(), superset_init=(),
    tol=1e-10, max_iter=100, rcond=1e-10
):
    p = len(groups)
    forbid = set() if forbid is None else set(forbid)
    S = set(S_init) - forbid
    superset = (set(superset_init) - forbid) | S

    def fit(S):
        S0 = sorted(S)
        B = compute_B_given_groups(S0, stat, groups, rcond)
        obj = compute_mse_obj_given_groups(B, S0, stat, gamma1, rho, lam)
        return B, obj

    B, obj = fit(S)

    for _ in range(max_iter):

        # Stage 1
        while True:
            improved = False
            for j in sorted(superset):
                if j in forbid: continue
                S_new = S - {j} if j in S else S | {j}
                B_new, obj_new = fit(S_new)

                if obj_new < obj - tol:
                    S, B, obj = S_new, B_new, obj_new
                    improved = True

            if not improved: break

        # Stage 2
        new_feature = False
        for j in range(p):
            if j in forbid: continue

            adding_new = j not in S and j not in superset
            S_new = S - {j} if j in S else S | {j}
            B_new, obj_new = fit(S_new)

            if obj_new < obj - tol:
                S, B, obj = S_new, B_new, obj_new
                if adding_new:
                    superset.add(j)
                    new_feature = True
                break

        if not new_feature: break

    S = sorted(map(int, S))
    return S, B, compute_mse_obj_given_groups(B, S, stat, gamma1, rho, lam)

### Baseline methods

#### Screening

In [14]:
def ridge_screening(X_list, y_list, k, lam=1.0, intercept=False):
    """
    X_list: list of X^e, each shape (n_e, p)
    y_list: list of y^e
    k: number of features to keep
    lam: ridge regularization
    """

    if intercept:
        k = k-1

    # Stack environments
    X_pool = np.vstack(X_list)
    y_pool = np.concatenate(y_list)
    _, p = X_pool.shape

    # Ridge solution
    XtX = X_pool.T @ X_pool
    Xty = X_pool.T @ y_pool
    beta_hat = np.linalg.solve(XtX + lam * np.eye(p), Xty)

    # Select top-k
    idx_sorted = np.argsort(np.abs(beta_hat))
    S_k = idx_sorted[-k:]

    if intercept:
        S_k = np.insert(S_k, 0, 0)

    return np.sort(S_k), beta_hat

#### LOEO

In [15]:
def loeo_mse(X_list, y_list, gamma1, rho, lam, s_max=None, verbose = 0):
    E = len(X_list)
    mse = []
    for e in range(E):
        X_train = [X_list[k] for k in range(E) if k != e]
        y_train = [y_list[k] for k in range(E) if k != e]
        # _, B, _ = vanilla_msetype(
        #     X_train, y_train, gamma1=gamma1, rho=rho,
        #     lam=lam, s_max=s_max, verbose=False
        # )
        stat = precompute_stat(X_train, y_train, n_list=None)
        inv_set, B, obj = cd_msetype(
            stat, lam=0.1, gamma1=5, rho=1, forbid=None, S_init=(), superset_init=(), tol=1e-10, max_iter=100
        )
        beta = B[0]
        mse.append(np.mean((y_list[e] - X_list[e] @ beta) ** 2))
        if verbose:
            print("finished", e, "in", E)
    return np.mean(mse)

#### ICP

In [16]:
from functools import reduce
import causalicp as icp

#### Causal Dantzig

In [17]:
# causal Danztig (aka hiddenICP) implementation in Python
# Code translated by ChatGPT from the Invariant Causal Prediction R package https://cran.r-project.org/web/packages/InvariantCausalPrediction/index.html
from scipy.stats import t, norm


def fit_hiddenICP(X, Y, ExpInd, alpha=0.05, mode="asymptotic", intercept=False):
    if isinstance(X, np.ndarray) and X.ndim == 1:
        X = X.reshape(-1, 1)
    if not isinstance(X, np.ndarray):
        raise ValueError("'X' must be a numpy array")
    if not isinstance(Y, np.ndarray):
        raise ValueError("'Y' must be a numpy array")
    if X.shape[0] <= X.shape[1]:
        raise ValueError(f"hiddenICP not suitable for high-dimensional data (at the moment) -- need nrow(X)>ncol(X) but have nrow(X)={X.shape[0]} and ncol(X)={X.shape[1]}")
    
    if not isinstance(ExpInd, list):
        if len(ExpInd) != len(Y):
            raise ValueError("if `ExpInd' is a vector, it needs to have the same length as `Y'")
        unique_env = np.unique(ExpInd)
        if len(unique_env) == 1:
            raise ValueError(f"there is just one environment ('ExpInd'={unique_env[0]} for all observations) and the method needs at least two distinct environments")
        if min(np.bincount(ExpInd)) <= 2:
            print("\n out put of 'table(ExpInd)':\n", np.bincount(ExpInd))
            raise ValueError("one environment has just one or two observations (as supplied by 'ExpInd'); there need to be at least 3 (and ideally dozens) of observations in each environment")
        
        K = len(unique_env)
        ExpIndNEW = []
        for uc in unique_env:
            ExpIndNEW.append(np.where(ExpInd == uc)[0])
        ExpInd = ExpIndNEW
    else:
        flat_indices = [idx for sublist in ExpInd for idx in sublist]
        if min(flat_indices) < 0:
            raise ValueError(f"if `ExpInd' is a list with indices of observations, minimal entry has to be at least 0 but is {min(flat_indices)}")
        if max(flat_indices) >= len(Y):
            raise ValueError(f"if `ExpInd' is a list with indices of observations, maximal entry has to be at most equal to {len(Y)-1} of the observations but is {max(flat_indices)}")
    
    colX = [f"Variable_{i}" for i in range(X.shape[1])]

    if intercept:
        # NOTE that the intercept is the last column
        X = np.hstack([X, np.ones((X.shape[0], 1))])
    
    K = len(ExpInd)
    n, p = X.shape

    KC = K if K > 2 else 1
    ConfInt = np.full((2, p), np.nan)
    pvalues = np.ones(p)
    
    betahatall = np.zeros(p)
    
    for kc in range(KC):
        ins = ExpInd[kc]
        out = np.setdiff1d(np.arange(n), ins)
        
        DS = (X[ins].T @ X[ins]) / len(ins) - (X[out].T @ X[out]) / len(out) #  (p, p)
        Drho = (X[ins].T @ Y[ins]) / len(ins) - (X[out].T @ Y[out]) / len(out) # (p,)
        try:
            DSI = np.linalg.inv(DS) # (p, p)
        except np.linalg.LinAlgError as e:
            raise ValueError(f"Matrix inversion failed: {e}")
        
        betahat = np.linalg.solve(DS, Drho) # (p,)
        betahatall += betahat
        
        
        Zin = np.zeros((len(ins), p))
        Zout = np.zeros((len(out), p))
        
        for i in range(len(ins)):
            tmp = DSI @ X[ins[i]] # (p,p) @ (p,) = (p,)
            Zin[i] = -tmp * np.sum(tmp * Drho) + Y[ins[i]] * tmp
        for i in range(len(out)):
            tmp = DSI @ X[out[i]]
            Zout[i] = -tmp * np.sum(tmp * Drho) + Y[out[i]] * tmp

        sigmavec = np.sqrt(np.diag(np.cov(Zin, rowvar=False) / len(ins) + np.cov(Zout, rowvar=False) / len(out)))
        
        pvalues = np.minimum(pvalues, 2 * K * (1 - t.cdf(np.abs(betahat) / np.maximum(1e-10, sigmavec), df=n - 1)))
        
        addvar = norm.ppf(max(0.5, 1 - alpha / (2 * K))) * sigmavec
        maximineffectsN = np.sign(betahat) * np.maximum(0, np.abs(betahat) - addvar)
        
        # Update lower bounds of confidence intervals
        new_lower_bounds = np.nan_to_num(betahat - addvar, nan=-np.inf)
        if kc == 0:
            ConfInt[0] = new_lower_bounds
        else:
            ConfInt[0] = np.minimum(ConfInt[0], new_lower_bounds)
        
        new_upper_bounds = np.nan_to_num(betahat + addvar, nan=np.inf)
        if kc == 0:
            ConfInt[1] = new_upper_bounds
        else:
            ConfInt[1] = np.maximum(ConfInt[1], new_upper_bounds)
        
        if kc == 0:
            maximineffects = maximineffectsN
        else:
            maximineffects = np.where(np.abs(maximineffectsN) > np.abs(maximineffects), maximineffectsN, maximineffects)
    
    betahat = betahatall / KC
    maximinCoefficients = maximineffects
    ConfInt = np.sort(ConfInt, axis=0) #[::-1]
    
    retobj = {
        "betahat": betahat,
        "maximinCoefficients": maximinCoefficients,
        "ConfInt": ConfInt,
        "pvalues": pvalues,
        "colnames": colX,
        "alpha": alpha
    }
    
    return retobj

#### EILLS

In [18]:
# Adapted from the official code repo https://github.com/wmyw96/EILLS
# Paper link: https://arxiv.org/pdf/2303.03092

import numpy as np
import heapq


def least_squares(X, y):
    cov_x = np.matmul(X.T, X)
    cov_xy = np.matmul(X.T, y)
    return np.squeeze(np.dot(np.linalg.inv(cov_x), cov_xy))


def pooled_least_squares(xs, ys):
    return least_squares(np.concatenate(xs, 0), np.concatenate(ys, 0))


def calc_fair_ll_loss(var_set, gamma, covs_xx, covs_xy, pen_coeff=0.0):
    num_envs = len(covs_xx)
    A = np.zeros((len(var_set), len(var_set)))
    b = np.zeros((len(var_set), 1))
    c = 0
    for i in range(num_envs):
        xx = covs_xx[i][var_set, :]
        xx = xx[:, var_set]
        xy = covs_xy[i][var_set, :]
        A = A + (1 + gamma) * xx / num_envs
        b = b + (1 + gamma) * xy / num_envs
        c = (
            c
            + gamma
            * np.matmul(np.matmul(np.transpose(xy), np.linalg.inv(xx)), xy)
            / num_envs
        )

    cur_beta = np.matmul(np.linalg.inv(A), b)
    cur_loss = (
        0.5 * np.matmul(np.matmul(np.transpose(cur_beta), A), cur_beta)
        - np.matmul(np.transpose(cur_beta), b)
        + 0.5 * c
        + pen_coeff * len(var_set)
    )
    return cur_beta, cur_loss


def calc_eills_loss(var_set, gamma, covs_xx, covs_xy, pen_coeff): 
    num_envs = len(covs_xx)
    A = np.zeros((len(var_set), len(var_set)))
    b = np.zeros((len(var_set), 1))
    c = 0
    for i in range(num_envs):
        xx = covs_xx[i][var_set, :]
        xx = xx[:, var_set]
        xy = covs_xy[i][var_set, :]
        A = A + xx / num_envs + gamma * np.matmul(xx, xx) / num_envs
        b = b + xy / num_envs + gamma * np.matmul(xx, xy) / num_envs
        c = c + gamma * np.matmul(np.transpose(xy), xy) / num_envs

    cur_beta = np.matmul(np.linalg.inv(A), b)
    cur_loss = (
        0.5 * np.matmul(np.matmul(np.transpose(cur_beta), A), cur_beta)
        - np.matmul(np.transpose(cur_beta), b)
        + 0.5 * c
        + pen_coeff * len(var_set)
    )
    return cur_beta, cur_loss


def varset_string_to_numpy(set_str):
    var_set = []
    for i in range(len(set_str)):
        if set_str[i] == "1":
            var_set.append(i)
    return np.array(var_set, dtype=np.int)


def set_to_string(var_set, dim_x):
    myset = set(var_set)
    set_str = ""
    for i in range(dim_x):
        set_str += str(int(i in myset))
    return set_str


def beta_local_to_global(str_set, cand_beta):
    global_beta = np.zeros((len(str_set), 1))
    local_index = 0
    for i in range(len(str_set)):
        if str_set[i] == "1":
            global_beta[i] = cand_beta[local_index]
            local_index += 1
    return global_beta


def support_set(beta):
    var_set = []
    for i in range(np.shape(beta)[0]):
        if np.abs(beta[i]) > 1e-6:
            var_set.append(i)
    return var_set


def greedy_search(
    x_list, y_list, gamma, iters=1000, loss_type="fair", cand_sets=[], show_log=True
):
    num_envs = len(x_list)
    covs_xx = []
    covs_xy = []
    covs_yy = []
    dim_x = np.shape(x_list[0])[1]

    if show_log:
        print(f"Linear model: greedy search, loss type = {loss_type}, d = {dim_x}")

    ns = 0
    for i in range(num_envs):
        x, y = x_list[i], y_list[i]
        n = np.shape(x)[0]
        ns += n
        print(f"Env = {i}, number of samples = {n}")
        covs_xx.append(np.matmul(np.transpose(x), x) / n)
        covs_xy.append(np.matmul(np.transpose(x), y) / n)
        covs_yy.append(np.matmul(np.transpose(y), y) / n)

    null_loss = sum([yy for yy in covs_yy]) / num_envs

    min_loss = 0
    min_beta = np.zeros((dim_x, 1))
    min_set = []

    losses, visited = {}, {}

    calc_loss = calc_fair_ll_loss
    if loss_type == "eills":
        calc_loss = calc_eills_loss

    pooled_lse = pooled_least_squares(x_list, y_list)
    set1 = ""
    for i in range(dim_x):
        set1 += str(int(np.abs(pooled_lse[i]) > 10 * np.log(dim_x) / ns))
    print(set1)
    pen_coeff = 6 * np.log(dim_x) / ns * num_envs

    init_sets = ["0" * dim_x, set1]
    for cand_set in cand_sets:
        init_sets.append(set_to_string(cand_set, dim_x))

    candidates_heap = []
    for a in init_sets:
        _, v = calc_loss(varset_string_to_numpy(a), gamma, covs_xx, covs_xy, pen_coeff)
        candidates_heap.append((float(v), a))
        losses[a] = v

    heapq.heapify(candidates_heap)

    for it in range(iters):
        _, cur_set = heapq.heappop(candidates_heap)
        visited[cur_set] = 1
        for j in range(dim_x):
            if cur_set[j] == "0":
                # index j not in current set, consider to incorporate it in
                cand_set = cur_set[:j] + "1" + cur_set[j + 1 :]
            else:
                # index j in current set, consider to eliminate it out
                cand_set = cur_set[:j] + "0" + cur_set[j + 1 :]

            if cand_set not in losses:
                cand_beta, cand_loss = calc_loss(
                    varset_string_to_numpy(cand_set), gamma, covs_xx, covs_xy, pen_coeff
                )
                losses[cand_set] = cand_loss
                heapq.heappush(
                    candidates_heap,
                    (cand_loss + 1.0 * (cand_loss > losses[cur_set]), cand_set),
                )
                if cand_loss < min_loss:
                    min_loss = cand_loss
                    min_set = varset_string_to_numpy(cand_set)
                    min_beta = beta_local_to_global(cand_set, cand_beta)

    print(
        f"Greedy search [gamma = {gamma}]: var_set = {min_set}, loss = {min_loss + null_loss}"
    )
    return np.squeeze(min_beta)


def brute_force(x_list, y_list, gamma, loss_type="fair", show_log=False, include_intercept=True):
    num_envs = len(x_list)
    covs_xx = []
    covs_xy = []
    covs_yy = []
    dim_x = np.shape(x_list[0])[1]
    if include_intercept:
        full_dim_x = dim_x + 1
    else:
        full_dim_x = dim_x
    
    if show_log:
        print(f"Linear model: brute force search, loss type = {loss_type}, d = {dim_x}")

    for i in range(num_envs):
        x, y = x_list[i], y_list[i]
        if y.ndim == 1:
            y = y.reshape(-1, 1)
        n = np.shape(x)[0]
        if show_log:
            print(f"Env = {i}, number of samples = {n}")
        if include_intercept:
            x = np.concatenate([x, np.ones((n, 1))], 1)
        covs_xx.append(np.matmul(np.transpose(x), x) / n)
        covs_xy.append(np.matmul(np.transpose(x), y) / n)
        covs_yy.append(np.matmul(np.transpose(y), y) / n)

    null_loss = sum([yy for yy in covs_yy]) / num_envs
    
    min_loss = 0
    min_beta = np.zeros((full_dim_x, 1))
    min_set = []

    calc_loss = calc_fair_ll_loss
    if loss_type == "eills":
        calc_loss = calc_eills_loss

    for sel in range(2**dim_x):
        var_set = []
        for j in range(dim_x):
            if sel & (2**j):
                var_set.append(j)
        if include_intercept:
            var_set.append(dim_x)

        if len(var_set) == 0:
            cur_loss = 0
            cur_beta = np.zeros((full_dim_x, 1))
        else:
            cur_beta, cur_loss = calc_loss(
                np.array(var_set, dtype=np.int32), gamma, covs_xx, covs_xy, 0
            )

        if cur_loss < min_loss:
            min_loss = cur_loss
            min_set = var_set
            min_beta = np.zeros((full_dim_x, 1))
            for i, idx in enumerate(var_set):
                min_beta[idx] = cur_beta[i]
    if include_intercept:
        min_set = min_set[:-1]
    
    if show_log:
        print(
            f"Brute force search [gamma = {gamma}]: var_set = {min_set}, loss = {min_loss + null_loss}"
        )
    min_set = np.array(min_set)
    return np.squeeze(min_beta), min_set


def fit_eills(x_list, y_list, true_para=None, show_log=False, include_intercept=True, gamma=36):
	return brute_force(x_list, y_list, gamma=gamma, 
                    loss_type='eills', include_intercept=include_intercept, show_log=show_log)

#### BIP

In [19]:
# BIP-s

from bip.core_utils import pi_analytical
from bip.utils import get_all_zs

#### NegDRO

In [20]:
import numpy as np
import torch

def compute_loss(x, y, b, intercept):
    """
    Compute the mean squared loss for a given environment.
    """
    if intercept:
        x = torch.cat([x, torch.ones(x.size(0), 1)], dim=1)
    preds = x @ b
    return torch.mean((y - preds) ** 2)    
        
def weight_solver_closed_form(losses, lambda_reg):
    """
    Given environments' losses, solve for the optimal weight using closed-form solution
    \argmax_{w\in \Delta^L} \ell^T w - \lambda \|w\|_2^2
    it has closed-form solution:
    w_l = max((\ell_l - mean(\ell)) / (2 * \lambda) + 1/L, 0)
    and then normalize w to sum to 1.
    """
    L = len(losses)
    losses_np = losses.detach().numpy()
    mean_loss = np.mean(losses_np)
    w_opt = (losses_np - mean_loss) / (2 * lambda_reg) + 1 / L
    w_opt = np.maximum(w_opt, 0)  # Ensure non-negativity
    if np.sum(w_opt) > 0:
        w_opt /= np.sum(w_opt)  # Normalize to sum to 1
    else:
        w_opt = np.ones(L) / L  # Fallback to uniform distribution if all weights are zero
    return torch.tensor(w_opt, dtype=torch.float32)

def negDRO(x_list, y_list, gamma, intercept=False, 
    lr_b=0.01, early_stop=False, 
    verbose=False, seed=None, num_iter=5000, log_interval=100):
    """
    Main algorithm for NegDRO (Negative Distributionally Robust Optimization)
    with ridge penalty and coefficient trajectory logging.

    Args:
        x_list (list of torch.Tensor or np.ndarray): Feature matrices for each environment.
        y_list (list of torch.Tensor or np.ndarray): Target vectors for each environment.
        gamma (float): Regularization parameter controlling the negative weighting.
        intercept (bool): If True, includes an intercept term in the model.
        lr_b (float): Learning rate for parameter b.
        early_stop (bool): If True, enables early stopping based on gradient norm.
        verbose (bool): If True, prints progress updates.
        seed (int, optional): Random seed for reproducibility.
        num_iter (int): Number of iterations for optimization.
        log_interval (int): Interval (in iterations) for logging coefficient trajectories.

    Returns:
        b_final (np.ndarray): Final coefficient vector.
        b_traj (list of np.ndarray): Logged coefficient trajectories during training.
    """

    # --------------------------------------------------------------------------
    # Initialization
    # --------------------------------------------------------------------------
    L = len(x_list)
    _, p = x_list[0].shape

    # Convert all inputs to float tensors
    x_list = [torch.tensor(x, dtype=torch.float32) for x in x_list]
    y_list = [torch.tensor(y, dtype=torch.float32) for y in y_list]

    # Set random seed
    if seed is not None:
        torch.manual_seed(seed)

    # Initialize coefficients
    b_dim = p + 1 if intercept else p
    b = torch.nn.Parameter(torch.randn(b_dim) * 0.5) # / torch.sqrt(torch.tensor(b_dim, dtype=torch.float32)))

    # Initialize uniform weights over environments
    w = torch.ones(L) / L
    w.requires_grad = False

    # Optimizer & scheduler
    optimizer_b = torch.optim.Adam([b], lr=lr_b)
    scheduler_b = torch.optim.lr_scheduler.StepLR(optimizer_b, step_size=1000, gamma=0.5)
    
    # Scheduler for lambda_reg
    lam_schedule_c=1.0           # c in c * M_est / sqrt(t + t0)
    lam_schedule_t0=100          # t0 to avoid overly large weights early on
    M_est=1.0                     # rough smoothness scale; can be tuned or estimated

    # Trackers
    min_grad_norm = float("inf")
    b_best = b.detach().clone()
    converged = False
    b_traj = []  # store coefficient trajectory

    # --------------------------------------------------------------------------
    # Optimization loop
    # --------------------------------------------------------------------------
    for i in range(num_iter):
        optimizer_b.zero_grad()

        # Compute per-environment losses
        losses = [compute_loss(x_list[e], y_list[e], b, intercept) for e in range(L)]
        losses_tensor = torch.stack(losses)
        
        lambda_reg_t = lam_schedule_c * M_est / np.sqrt(i + lam_schedule_t0)

        # Solve for environment weights in closed form
        w = weight_solver_closed_form(losses_tensor.detach(), lambda_reg_t)
        w.requires_grad = False

        # Compute total objective
        total_loss = torch.sum((w - gamma / (1 + gamma * L)) * losses_tensor) \
                     - lambda_reg_t * torch.sum(w ** 2)

        # Backpropagation
        total_loss.backward()
        grad_norm = torch.norm(b.grad).item()

        # Track best (lowest-gradient) b
        if grad_norm < min_grad_norm:
            min_grad_norm = grad_norm
            b_best = b.detach().clone()

        # Early stopping (optional)
        if early_stop and grad_norm < 1e-4:
            converged = True
            if verbose:
                print(f"[Early Stop] Iter {i:4d} | Loss={total_loss.item():.4f} | GradNorm={grad_norm:.5e}")
            break

        # Safety stop if gradient explodes
        if grad_norm > 5e3:
            print("⚠️ Gradient explosion detected. Stopping early.")
            break

        # Update parameters
        optimizer_b.step()
        scheduler_b.step()

        # Log coefficient trajectory
        if (i % log_interval == 0) or (i == num_iter - 1):
            b_traj.append(b.detach().cpu().numpy().flatten())
            if verbose:
                print(f"Iter {i:4d} | Loss={total_loss.item():.4f} | GradNorm={grad_norm:.5e}")

    # --------------------------------------------------------------------------
    # Finalize output
    # --------------------------------------------------------------------------
    if not converged:
        b = b_best

    b_final = b.detach().cpu().numpy().flatten()
    return b_final, b_traj

#### CoCo

In [21]:
import numpy as np
import torch


def fit_coco_linear_regularized(
    X_list,
    y_list,
    n_list=None,
    lam=1.0,
    lr=1e-2,
    max_iter=10000,
    include_intercept=True,
    equal_env_weight=True,
    penalty_sqrt=False,
    l2=0.0,
    dtype=torch.float64,
    device=None,
    seed=None,
    verbose=False,
    print_every=1000,
):
    """
    Fit regularized linear CoCo:

        risk(theta) + lam * coco_penalty(theta)

    where

        risk(theta)
            = average_e 1/(2 n_e) ||y_e - X_e theta||^2,

        coco_penalty(theta)
            = average_e [g_{e,0}^2 + sum_j (beta_j g_{e,j})^2],

    and

        g_e = grad_theta R_e(theta).

    Parameters
    ----------
    X_list : list of arrays
        X_list[e] has shape (n_e, p).
    y_list : list of arrays
        y_list[e] has shape (n_e,).
    n_list : list of int, optional
        Sample sizes. If None, inferred from X_list.
    lam : float
        CoCo regularization strength.
    lr : float
        Adam learning rate.
    max_iter : int
        Number of optimization iterations.
    include_intercept : bool
        If True, include an intercept.
    equal_env_weight : bool
        If True, each environment receives equal weight.
        If False, environments are weighted by sample size.
    penalty_sqrt : bool
        If True, use sqrt(coco_penalty).
        If False, use coco_penalty directly.
    l2 : float
        Optional ridge penalty on slopes only.
    dtype : torch dtype
    device : torch device
    seed : int or None
    verbose : bool
    print_every : int

    Returns
    -------
    result : dict
        Contains beta, intercept, theta, objective, risk, coco_penalty.
    """

    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)

    if device is None:
        device = torch.device("cpu")

    E = len(X_list)

    X_tensors = []
    y_tensors = []

    for X, y in zip(X_list, y_list):
        X = torch.as_tensor(X, dtype=dtype, device=device)
        y = torch.as_tensor(y, dtype=dtype, device=device).reshape(-1)

        if include_intercept:
            ones = torch.ones((X.shape[0], 1), dtype=dtype, device=device)
            X = torch.cat([ones, X], dim=1)

        X_tensors.append(X)
        y_tensors.append(y)

    if n_list is None:
        n_list = [X.shape[0] for X in X_tensors]

    if equal_env_weight:
        env_weights = torch.ones(E, dtype=dtype, device=device) / E
    else:
        n_total = float(sum(n_list))
        env_weights = torch.tensor(
            [n_e / n_total for n_e in n_list],
            dtype=dtype,
            device=device,
        )

    p_aug = X_tensors[0].shape[1]

    # Initialize at pooled OLS when possible. This is usually more stable than random init.
    X_pool = torch.cat(X_tensors, dim=0)
    y_pool = torch.cat(y_tensors, dim=0)

    try:
        theta_init = torch.linalg.solve(
            X_pool.T @ X_pool + 1e-6 * torch.eye(p_aug, dtype=dtype, device=device),
            X_pool.T @ y_pool,
        )
    except RuntimeError:
        theta_init = torch.zeros(p_aug, dtype=dtype, device=device)

    theta = torch.nn.Parameter(theta_init.clone())

    optimizer = torch.optim.Adam([theta], lr=lr)

    history = []

    for it in range(max_iter):
        optimizer.zero_grad()

        risk = 0.0
        coco_penalty = 0.0

        for w_e, X_e, y_e in zip(env_weights, X_tensors, y_tensors):
            n_e = X_e.shape[0]

            resid_e = X_e @ theta - y_e

            # Environment-specific squared-loss risk:
            # R_e(theta) = 1/(2n_e) ||X_e theta - y_e||^2
            risk_e = 0.5 * torch.mean(resid_e ** 2)

            # Analytic gradient of R_e(theta):
            # g_e = X_e^T (X_e theta - y_e) / n_e
            grad_e = X_e.T @ resid_e / n_e

            if include_intercept:
                # Penalize intercept score directly.
                intercept_score = grad_e[0] ** 2

                # Penalize slope scores multiplied by slope coefficients.
                slope_score = torch.sum((theta[1:] * grad_e[1:]) ** 2)

                coco_e = intercept_score + slope_score
            else:
                coco_e = torch.sum((theta * grad_e) ** 2)

            risk = risk + w_e * risk_e
            coco_penalty = coco_penalty + w_e * coco_e

        if penalty_sqrt:
            coco_term = torch.sqrt(coco_penalty + 1e-12)
        else:
            coco_term = coco_penalty

        if include_intercept:
            ridge_term = l2 * torch.sum(theta[1:] ** 2)
        else:
            ridge_term = l2 * torch.sum(theta ** 2)

        objective = risk + lam * coco_term + ridge_term

        objective.backward()
        optimizer.step()

        history.append(
            {
                "objective": float(objective.detach().cpu()),
                "risk": float(risk.detach().cpu()),
                "coco_penalty": float(coco_penalty.detach().cpu()),
            }
        )

        if verbose and it % print_every == 0:
            print(
                f"iter={it:5d}, "
                f"obj={history[-1]['objective']:.6g}, "
                f"risk={history[-1]['risk']:.6g}, "
                f"coco={history[-1]['coco_penalty']:.6g}"
            )

    theta_hat = theta.detach().cpu().numpy()

    if include_intercept:
        intercept = theta_hat[0]
        beta = theta_hat[1:]
    else:
        intercept = 0.0
        beta = theta_hat

    return {
        "beta": beta,
        "intercept": intercept,
        "theta": theta_hat,
        "objective": history[-1]["objective"],
        "risk": history[-1]["risk"],
        "coco_penalty": history[-1]["coco_penalty"],
        "history": history,
    }

#### IRM

In [22]:
import numpy as np
import torch
from torch import nn, optim, autograd


def irm_linear(
    X_list, y_list,
    penalty_weight=1e4,
    penalty_anneal_iters=100,
    l2_weight=1e-3,
    lr=1e-3,
    steps=500,
    device=None
):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    X = [torch.tensor(x, dtype=torch.float32, device=device) for x in X_list]
    y = [torch.tensor(v, dtype=torch.float32, device=device).view(-1, 1) for v in y_list]

    p = X[0].shape[1]
    model = nn.Linear(p, 1, bias=False).to(device)
    nn.init.xavier_uniform_(model.weight)

    def penalty(pred, target):
        scale = torch.tensor(1., device=device, requires_grad=True)
        loss = ((pred * scale - target) ** 2).mean()
        grad = autograd.grad(loss, scale, create_graph=True)[0]
        return grad.pow(2)

    optimizer = optim.Adam(model.parameters(), lr=lr)

    for step in range(steps):
        risks, penalties = [], []

        for Xe, ye in zip(X, y):
            pred = model(Xe)
            risks.append(((pred - ye) ** 2).mean())
            penalties.append(penalty(pred, ye))

        risk = torch.stack(risks).mean()
        irm_penalty = torch.stack(penalties).mean()
        l2 = sum(w.pow(2).sum() for w in model.parameters())

        pw = penalty_weight if step >= penalty_anneal_iters else 1.0
        loss = risk + l2_weight * l2 + pw * irm_penalty
        if pw > 1: loss /= pw

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return model.weight.detach().cpu().numpy().reshape(-1)

#### Wasserstein

In [572]:
from WVM.py_implementation.wasserstein_variance import DirectWassersteinVariance, WV_screening

#### FAIR

In [1571]:
from copy import deepcopy

from FAIR.methods.fair_algo import FairMLP, FairGumbelAlgo
from FAIR.utils import aos_default_hyper_params


def fit_fair_nn(X_list, y_list, gamma=36, niters=10000,
                threshold=.9, anneal_rate=.96, seed=0):
    np.random.seed(seed)
    torch.manual_seed(seed)
    E, p = len(X_list), X_list[0].shape[1]
    xs = [np.asarray(X, dtype=np.float32) for X in X_list]
    ys = [np.asarray(y, dtype=np.float32).reshape(-1, 1) for y in y_list]
    hp = deepcopy(aos_default_hyper_params)
    hp.update(
        offset=-1, niters=niters,
        init_temp=5., final_temp=.1,
        anneal_iter=100, anneal_rate=anneal_rate
    )
    model = FairMLP(E, p, depth_g=1, width_g=128, depth_f=2, width_f=196)
    loss = lambda yhat, y: .5 * torch.mean((yhat - y)**2)
    algo = FairGumbelAlgo(E, p, model, gamma, loss, hp)
    result = algo.run_gumbel(
        (xs, ys), eval_iter=max(1, niters//10),
        gate_samples=20, log=False
    )
    probs = np.asarray(result["gate_rec"][-1])
    support = np.where(probs > threshold)[0]
    return support, probs, np.asarray(result["gate_rec"])

### Data generation

#### all interventions

In [23]:
def generate_synthetic_data_all_intervention(
    n, p, E, p_act, lb, ub, lb_star, ub_star,
    sigma_x_min, sigma_x_max, sigma_y,
    p_star_min, p_star_max, pos_y=None
):
    total = p + 1
    pos_y = total // 2 if pos_y is None else pos_y

    # Common topological order
    perm = np.random.permutation(total)
    i_y = np.where(perm == p)[0][0]
    perm[i_y], perm[pos_y] = perm[pos_y], perm[i_y]

    # Invariant parent set and coefficients of Y
    candidates = perm[:pos_y]
    s = np.random.randint(p_star_min, min(p_star_max, len(candidates)) + 1)
    S_star = np.sort(np.random.choice(candidates, s, replace=False))
    beta_star = np.zeros(p)
    beta_star[S_star] = (
        np.random.choice([-1, 1], s)
        * np.random.uniform(lb_star, ub_star, s)
    )
    X_list, y_list = [], []

    for _ in range(E):
        alpha = np.zeros((total, total))
        # Environment-specific X mechanisms; invariant Y mechanism
        for i in range(total):
            child = perm[i]
            for parent in perm[:i]:
                if child == p:
                    if parent in S_star: 
                        alpha[child, parent] = beta_star[parent]
                elif np.random.rand() < p_act:
                    alpha[child, parent] = (
                        np.random.choice([-1, 1])
                        * np.random.uniform(lb, ub)
                    )
        sigma_x = np.random.uniform(sigma_x_min, sigma_x_max, p)
        data = np.zeros((n, total))
        for t in range(n):
            values = np.zeros(total)
            for node in perm:
                sd = sigma_y if node == p else sigma_x[node]
                values[node] = alpha[node] @ values + np.random.normal(0, sd)
            data[t] = values
        X_list.append(data[:, :p])
        y_list.append(data[:, p])

    return S_star, beta_star, X_list, y_list, [n] * E

#### soft interventions

In [24]:
def generate_synthetic_data_soft_intervention(
    n, p, E, p_act, lb, ub, lb_star, ub_star,
    sigma_x_min, sigma_x_max, sigma_y,
    p_star_min, p_star_max, p_resample=1.0,
    pos_y=None, intervene_on_Y=False, random_state=None
):
    rng = np.random.default_rng(random_state)
    coef = lambda a, b: rng.choice([-1., 1.]) * rng.uniform(a, b)

    total = p + 1
    pos_y = total // 2 if pos_y is None else pos_y
    if not 0 <= pos_y < total: raise ValueError("pos_y must be in {0,...,p}.")

    # Common topological order
    perm = rng.permutation(total)
    i_y = np.where(perm == p)[0][0]
    perm[i_y], perm[pos_y] = perm[pos_y], perm[i_y]

    # Invariant parents and coefficients of Y
    candidates = perm[:pos_y]
    if len(candidates) == 0: raise ValueError("No candidates available before Y.")

    s_min, s_max = min(p_star_min, len(candidates)), min(p_star_max, len(candidates))
    if s_min < 1: raise ValueError("Need at least one parent candidate for Y.")

    s = rng.integers(s_min, s_max + 1)
    S_star = np.sort(rng.choice(candidates, s, replace=False))

    beta_star = np.zeros(p)
    beta_star[S_star] = [coef(lb_star, ub_star) for _ in S_star]

    # Fixed DAG support
    support = np.zeros((total, total), int)
    support[p, S_star] = 1

    for i, child in enumerate(perm):
        if child == p: continue
        for parent in perm[:i]:
            if rng.random() < p_act: support[child, parent] = 1

    # Baseline coefficients
    alpha_base = np.zeros((total, total))
    alpha_base[p, S_star] = beta_star[S_star]

    for child in range(p):
        for parent in np.where(support[child])[0]:
            alpha_base[child, parent] = coef(lb, ub)

    # Same noise distributions in every environment
    sigma_x = rng.uniform(sigma_x_min, sigma_x_max, p)

    X_list, y_list = [], []

    for e in range(E):
        alpha = alpha_base.copy()

        if e > 0:
            for child in range(p):
                for parent in np.where(support[child])[0]:
                    if rng.random() < p_resample:
                        alpha[child, parent] = coef(lb, ub)

            if intervene_on_Y:
                for parent in S_star:
                    if rng.random() < p_resample:
                        alpha[p, parent] = coef(lb_star, ub_star)

        data = np.zeros((n, total))

        for t in range(n):
            v = np.zeros(total)
            for node in perm:
                sd = sigma_y if node == p else sigma_x[node]
                v[node] = alpha[node] @ v + rng.normal(0, sd)
            data[t] = v

        X_list.append(data[:, :p])
        y_list.append(data[:, p])

    return S_star, beta_star, X_list, y_list, [n] * E

#### identification

In [28]:
def generate_X1_to_Y_to_X2_example(
    n_list=(1000, 1000),
    k_independent=0,
    gamma=1.0,
    tau2=1.0,
    sigma_eps2=1.0,
    a_list=(1.0, 0.5),
    v_list=(2.0, 1.5),
    z_var=1.0,
    seed=None,
    return_params=False,
):
    """
    Generate multi-environment data from the SEM

        X1^e = N1^e
        Y^e  = gamma * X1^e + eps^e
        X2^e = a_e * Y^e + eta^e

    plus k_independent independent noise features

        Z_1^e, ..., Z_k^e,

    where the Z variables are independent of X1, Y, X2 and all structural noises.

    The default two-environment parameters are chosen so that for S = {X2},

        beta^{1,S} = beta^{2,S},

    but

        sigma^{1,S} != sigma^{2,S}.

    Parameters
    ----------
    n_list : tuple or list of int
        Sample sizes for each environment.

    k_independent : int
        Number of additional independent noise features.

    gamma : float
        Structural coefficient from X1 to Y.

    tau2 : float
        Variance of X1.

    sigma_eps2 : float
        Variance of eps in the Y equation.

    a_list : tuple or list of float
        Coefficients from Y to X2 in each environment.

    v_list : tuple or list of float
        Noise variances of eta in each environment.

    z_var : float
        Variance of each independent noise feature.

    seed : int, optional
        Random seed.

    return_params : bool
        If True, also return the parameter dictionary.

    Returns
    -------
    X_list : list of np.ndarray
        X_list[e] has shape (n_e, 2 + k_independent).
        Column 0 is X1.
        Column 1 is X2.
        Columns 2 onward are independent noise features.

    y_list : list of np.ndarray
        y_list[e] has shape (n_e,).

    n_list : list of int
        Sample sizes.

    params : dict, optional
        Returned only if return_params=True.
    """

    rng = np.random.default_rng(seed)

    n_list = list(n_list)
    a_list = list(a_list)
    v_list = list(v_list)

    E = len(n_list)

    if len(a_list) != E:
        raise ValueError("a_list must have the same length as n_list.")
    if len(v_list) != E:
        raise ValueError("v_list must have the same length as n_list.")
    if k_independent < 0:
        raise ValueError("k_independent must be nonnegative.")
    if z_var < 0:
        raise ValueError("z_var must be nonnegative.")

    X_list = []
    y_list = []

    for e in range(E):
        n = int(n_list[e])
        a_e = a_list[e] + rng.normal(loc=0.0, scale=0.05, size=1)
        v_e = v_list[e] + rng.normal(loc=0.0, scale=0.05, size=1)

        X1 = rng.normal(loc=0.0, scale=np.sqrt(tau2), size=n)
        eps = rng.normal(loc=0.0, scale=np.sqrt(sigma_eps2), size=n)
        eta = rng.normal(loc=0.0, scale=np.sqrt(v_e), size=n)

        y = gamma * X1 + eps
        X2 = a_e * y + eta

        if k_independent > 0:
            Z = rng.normal(
                loc=0.0,
                scale=np.sqrt(z_var),
                size=(n, k_independent),
            )
            X = np.column_stack([X1, X2, Z])
        else:
            X = np.column_stack([X1, X2])

        X_list.append(X)
        y_list.append(y)

    if not return_params:
        return X_list, y_list, n_list

    c = gamma**2 * tau2 + sigma_eps2

    params = {
        "gamma": gamma,
        "tau2": tau2,
        "sigma_eps2": sigma_eps2,
        "a_list": a_list,
        "v_list": v_list,
        "k_independent": k_independent,
        "z_var": z_var,
        "feature_names": ["X1", "X2"] + [f"Z{j + 1}" for j in range(k_independent)],
        "true_parent_set": [0],
        "bad_set_equal_beta_unequal_sigma": [1],
        "independent_feature_set": list(range(2, 2 + k_independent)),
        "population_beta_on_X2": [
            (a_list[e] * c) / (a_list[e] ** 2 * c + v_list[e])
            for e in range(E)
        ],
        "population_residual_var_on_X2": [
            c - (a_list[e] * c) ** 2 / (a_list[e] ** 2 * c + v_list[e])
            for e in range(E)
        ],
    }

    return X_list, y_list, n_list, params

#### Example/teaser

In [347]:
def generate_multiple_invariant(
    n_train, n_test, k=5, noise_std=.5,
    lbd=.5, ubd=2., perturb_strength=3, seed=None
):
    """
    SEM:
        X2 -> X3
        X1 -> Y <- X3
        Y  -> X4

    Environments:
        0: observational
        1: intervene on X1
        2: intervene on X2

    True parent set of Y:
        S* = {X1, X3}
    """
    rng = np.random.default_rng(seed)

    a3 = rng.uniform(lbd, ubd)   # X2 -> X3
    b1 = rng.uniform(lbd, ubd)   # X1 -> Y
    b3 = rng.uniform(lbd, ubd)   # X3 -> Y
    c4 = rng.uniform(lbd, ubd)   # Y  -> X4

    coef = np.zeros(4+k)
    coef[0], coef[2] = b1, b3

    def generate_env_data(n, env):
        X1 = rng.normal(0, 1, n)
        X2 = rng.normal(0, 1, n)

        if env == 1:
            X1 = rng.choice([-perturb_strength, perturb_strength], n)
        elif env == 2:
            X2 = rng.choice([-perturb_strength, perturb_strength], n)

        e3 = rng.normal(0, noise_std, n)
        ey = rng.normal(0, noise_std, n)
        e4 = rng.normal(0, noise_std, n)

        X3 = a3*X2 + e3
        Y  = b1*X1 + b3*X3 + ey
        X4 = c4*Y + e4

        X = np.column_stack([X1, X2, X3, X4])
        if k > 0: X = np.column_stack([X, rng.normal(0, 1, (n, k))])

        return X, Y

    X_train_list, y_train_list, n_train_list = [], [], []
    X_test_list, y_test_list, n_test_list = [], [], []

    for env in range(3):
        Xtr, ytr = generate_env_data(n_train, env)
        Xte, yte = generate_env_data(n_test, env)

        X_train_list.append(Xtr); y_train_list.append(ytr); n_train_list.append(n_train)
        X_test_list.append(Xte); y_test_list.append(yte); n_test_list.append(n_test)

    return (
        X_train_list, y_train_list, n_train_list,
        X_test_list, y_test_list, n_test_list, coef
    )

#### hidden confounding

In [942]:
def generate_hidden_sem_data(
    n, E, p_a=None, p_s=4, p_d=None, k=8, q=2,
    beta=None, Gamma_A=None, B_AS=None, Gamma_S=None,
    Gamma_D=None, phi_Y=None, theta_D=None,
    a_scales=None, s_scales=None, d_scales=None,
    sigma_y=.2, gamma_a_scale=.3, gamma_s_scale=.3,
    gamma_d_scale=.3, b_as_scale=.3, seed=None
):
    rng = np.random.default_rng(seed)

    p_a = rng.choice([2,3]) if p_a is None else int(p_a)
    p_d = rng.choice([2,3]) if p_d is None else int(p_d)
    n_list = [int(n)]*E if np.isscalar(n) else list(map(int, n))
    if len(n_list) != E: raise ValueError("n must have length E.")

    beta = rng.normal(0,1,p_s) if beta is None else np.asarray(beta,float)
    Gamma_A = rng.normal(0,gamma_a_scale,(q,p_a)) if Gamma_A is None else np.asarray(Gamma_A,float)
    B_AS = rng.normal(0,b_as_scale,(p_a,p_s)) if B_AS is None else np.asarray(B_AS,float)
    Gamma_S = rng.normal(0,gamma_s_scale,(q,p_s)) if Gamma_S is None else np.asarray(Gamma_S,float)
    Gamma_D = rng.normal(0,gamma_d_scale,(q,p_d)) if Gamma_D is None else np.asarray(Gamma_D,float)
    phi_Y = rng.normal(0,1,q) if phi_Y is None else np.asarray(phi_Y,float)
    theta_D = rng.normal(0,1,p_d) if theta_D is None else np.asarray(theta_D,float)

    if beta.shape != (p_s,): raise ValueError("beta has wrong shape.")
    if Gamma_A.shape != (q,p_a): raise ValueError("Gamma_A has wrong shape.")
    if B_AS.shape != (p_a,p_s): raise ValueError("B_AS has wrong shape.")
    if Gamma_S.shape != (q,p_s): raise ValueError("Gamma_S has wrong shape.")
    if Gamma_D.shape != (q,p_d): raise ValueError("Gamma_D has wrong shape.")
    if phi_Y.shape != (q,): raise ValueError("phi_Y has wrong shape.")
    if theta_D.shape != (p_d,): raise ValueError("theta_D has wrong shape.")

    a_scales = [1+.1*e for e in range(E)] if a_scales is None else list(a_scales)
    s_scales = [1+.1*e for e in range(E)] if s_scales is None else list(s_scales)
    d_scales = [1+.1*e for e in range(E)] if d_scales is None else list(d_scales)
    if not all(len(x) == E for x in [a_scales,s_scales,d_scales]):
        raise ValueError("Noise-scale lists must have length E.")

    X_list, y_list = [], []

    for e, ne in enumerate(n_list):
        H = rng.normal(0,1,(ne,q))

        X_A = H@Gamma_A + rng.normal(0,a_scales[e],(ne,p_a))
        X_S = X_A@B_AS + H@Gamma_S + rng.normal(0,s_scales[e],(ne,p_s))
        y = X_S@beta + H@phi_Y + rng.normal(0,sigma_y,ne)
        X_D = y[:,None]*theta_D + H@Gamma_D + rng.normal(0,d_scales[e],(ne,p_d))
        X_I = rng.normal(0,1,(ne,k))

        X_list.append(np.c_[X_A, X_S, X_D, X_I])
        y_list.append(y)

    A = np.arange(p_a)
    S = np.arange(p_a, p_a+p_s)
    D = np.arange(p_a+p_s, p_a+p_s+p_d)
    I = np.arange(p_a+p_s+p_d, p_a+p_s+p_d+k)

    true_coef = np.zeros(p_a+p_s+p_d+k)
    true_coef[S] = beta

    params = {
        "Gamma_A": Gamma_A,
        "B_AS": B_AS,
        "Gamma_S": Gamma_S,
        "Gamma_D": Gamma_D,
        "phi_Y": phi_Y,
        "theta_D": theta_D,
        "a_scales": np.asarray(a_scales),
        "s_scales": np.asarray(s_scales),
        "d_scales": np.asarray(d_scales)
    }

    return X_list, y_list, n_list, A, S, D, I, true_coef, params

#### all interventions nonlinear

In [1603]:
def generate_synthetic_data_all_intervention_nonlinear(
    n, p, E, p_act, lb, ub, lb_star, ub_star,
    sigma_x_min, sigma_x_max, sigma_y,
    p_star_min, p_star_max, pos_y=None,
    random_state=None
):
    rng = np.random.default_rng(random_state)
    coef = lambda a, b: rng.choice([-1., 1.]) * rng.uniform(a, b)

    total = p + 1
    pos_y = total // 2 if pos_y is None else pos_y
    if not 0 <= pos_y < total: raise ValueError("pos_y must be in {0,...,p}.")

    # Common topological order
    perm = rng.permutation(total)
    i_y = np.where(perm == p)[0][0]
    perm[i_y], perm[pos_y] = perm[pos_y], perm[i_y]

    # Invariant parent set and nonlinear mechanism of Y
    candidates = perm[:pos_y]
    s_min = min(p_star_min, len(candidates))
    s_max = min(p_star_max, len(candidates))
    if s_min < 1: raise ValueError("Need at least one parent candidate for Y.")

    s = rng.integers(s_min, s_max + 1)
    S_star = np.sort(rng.choice(candidates, s, replace=False))

    beta_star = np.zeros(p)
    beta_star[S_star] = [coef(lb_star, ub_star) for _ in S_star]

    X_list, y_list = [], []

    for _ in range(E):
        alpha = np.zeros((total, total))

        # Environment-specific X mechanisms; invariant Y mechanism
        for i, child in enumerate(perm):
            for parent in perm[:i]:
                if child == p:
                    if parent in S_star:
                        alpha[child, parent] = beta_star[parent]
                elif rng.random() < p_act:
                    alpha[child, parent] = coef(lb, ub)

        # Environment-specific X noise scales
        sigma_x = rng.uniform(sigma_x_min, sigma_x_max, p)

        data = np.zeros((n, total))

        for t in range(n):
            v = np.zeros(total)

            for node in perm:
                pa = np.flatnonzero(alpha[node])
                mu = np.sum(alpha[node, pa] * np.tanh(2 * np.sin(v[pa]))) if len(pa) else 0.
                sd = sigma_y if node == p else sigma_x[node]
                v[node] = mu + rng.normal(0, sd)

            data[t] = v

        X_list.append(data[:, :p])
        y_list.append(data[:, p])

    return S_star, beta_star, X_list, y_list, [n] * E

### Exe

#### baseline

In [ ]:
# strong interventions

RS = np.zeros((0, 8))

for p in [5, 10, 15, 20, 25, 30, 35, 40, 50, 60, 70, 80, 90, 100]:
    for l in range(10):
        n=500
        E_train=10
        if int(0.4 * p) < 16:   
            p_star = int(0.4 * p)
        else:
            p_star = 16 + int(0.1 * p)
        true_invariant, coef, X_list, y_list, n_list = generate_synthetic_data_all_intervention(
            n=n,
            p=p,
            E=E_train,
            p_act=np.random.choice([0.1,0.15]),
            lb=0.2, ub=0.5,
            lb_star=0.5, ub_star=0.5,
            sigma_x_min=0.5,
            sigma_x_max=1.0,
            sigma_y=0.5,
            p_star_min=p_star,
            p_star_max=p_star
        )
        super_structure_10 = ridge_screening(X_list, y_list, 10, lam=1.0, intercept=False)[0]
        super_structure_15 = ridge_screening(X_list, y_list, 15, lam=1.0, intercept=False)[0]


        # MIP-cd
        t1 = time.time()
        stat = precompute_stat(X_list, y_list, n_list=None)
        inv_set, B, obj = cd_msetype(
            stat, lam=0.01, gamma1=5, rho=1, forbid=None, S_init=(), superset_init=(), tol=1e-10, max_iter=100
        )
        t2 = time.time()
        coef_hat = B[0]
        rs = np.array([
            p,l,
            "MIP-cd",
            (np.linalg.norm(coef_hat - coef)).item(),
            set(inv_set).issuperset(set(true_invariant)),
            set(inv_set) == set(true_invariant),
            len(inv_set),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished MIP")
        

        # EILLS
        X_list_screen = []
        for e in range(len(X_list)):
            X_list_screen.append( X_list[e][:,super_structure_15] )
        t1 = time.time()
        coef_eills, g_eills = fit_eills(X_list_screen, y_list, include_intercept=False, gamma=20)
        t2 = time.time()
        if len(g_eills) == 0:
            inv_set = []
        else:
            inv_set = super_structure_15[g_eills] 
        coef_hat = np.zeros(p)
        coef_hat[super_structure_15] = coef_eills
        rs = np.array([
            p,l,
            "EILLS",
            (np.linalg.norm(coef_hat - coef)).item(),
            set(inv_set).issuperset(set(true_invariant)),
            set(inv_set) == set(true_invariant),
            len(inv_set),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished EILLS")


        # ICP
        X_list_screen = []
        for e in range(len(X_list)):
            X_list_screen.append( X_list[e][:,super_structure_10] )
        data = []
        for i in range(len(X_list_screen)):
            data.append( np.column_stack([X_list_screen[i], y_list[i]]) )
        target = X_list_screen[0].shape[1]
        t1 = time.time()
        result = icp.fit(data, target, alpha=0.05, precompute=True, verbose=False, color=False)
        t2 = time.time()
        if len(result.accepted_sets) == 0:
            intersection = []
        else:
            intersection = reduce(lambda a, b: a & b, result.accepted_sets)
        inv_set = super_structure_10[list(intersection)]
        rs = np.array([
            p,l,
            "ICP",
            np.nan,
            set(inv_set).issuperset(set(true_invariant)),
            set(inv_set) == set(true_invariant),
            len(inv_set),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished ICP")


        # Hidden-ICP
        X = np.vstack(X_list)      # shape (n_total, p)
        Y = np.concatenate(y_list) # shape (n_total,)
        ExpInd = []
        start = 0
        for i, Xi in enumerate(X_list):
            n_i = Xi.shape[0]
            ExpInd.extend([i]*n_i)
        ExpInd = np.array(ExpInd)  # length n_total
        t1 = time.time()
        result = fit_hiddenICP(X, Y, ExpInd, alpha=0.1, intercept=False)['maximinCoefficients']
        t2 = time.time()
        inv_set = np.where( result != 0 )[0]
        coef_hat = result
        rs = np.array([
            p,l,
            "Causal Dantzig",
            (np.linalg.norm(coef_hat - coef)).item(),
            set(inv_set).issuperset(set(true_invariant)),
            set(inv_set) == set(true_invariant),
            len(inv_set),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished Dantzig")


        # BIP-s
        X_list_screen = []
        for e in range(len(X_list)):
            X_list_screen.append( X_list[e][:,super_structure_10] )
        xs = []
        ys = []
        for i in range(len(X_list_screen)):
            xs.append(torch.tensor(X_list_screen[i]))
            ys.append(torch.tensor(y_list[i]))
        t1 = time.time()
        ols_result, log_posterior, posterior = pi_analytical(
            zs=None,        # None = enumerate all subsets automatically
            xs=xs,          # list of tensors or (E, N, p) tensor
            ys=ys,          # list of tensors or (E, N) tensor
            p_max=20,     # optional: restrict to subsets of size <= p_max
            epsilon=1e-4,   # ridge jitter for OLS stability
            use_vectorize=True,         # True = faster batched computation
            include_intercept=False,     # add intercept column to X
            keep_intercept=False         # always include intercept in z
        )
        t2 = time.time()
        p_ = xs[0].shape[-1]
        zs = get_all_zs(p_, p_max=10)   # all candidate z's in the same order as posterior
        map_idx = posterior.argmax()
        map_z = zs[map_idx]  
        inv_set = super_structure_10[np.where(map_z !=0 )[0]]
        coef_hat = np.zeros(p)
        coef_hat[super_structure_10] = ols_result[5][map_idx]
        rs = np.array([
            p,l,
            "BIP-s",
            (np.linalg.norm(coef_hat - coef)).item(),
            set(inv_set).issuperset(set(true_invariant)),
            set(inv_set) == set(true_invariant),
            len(inv_set),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished BIP")

        
        # NegDRO
        t1 = time.time()
        b_final, _ = negDRO(X_list, y_list, gamma = 20)
        t2 = time.time()
        coef_hat = b_final
        rs = np.array([
            p,l,
            "NegDRO",
            (np.linalg.norm(coef_hat - coef)).item(),
            np.nan,
            np.nan,
            np.nan,
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished NegDRO")


        # CoCo
        t1 = time.time()
        res = fit_coco_linear_regularized(
            X_list=X_list,
            y_list=y_list,
            n_list=n_list,
            lam=10.0,
            lr=1e-2,
            max_iter=10000,
            include_intercept=False,
            equal_env_weight=True,
            penalty_sqrt=False,
            verbose=False,
        )
        t2 = time.time()
        coef_hat = res["beta"]
        rs = np.array([
            p,l,
            "CoCo",
            (np.linalg.norm(coef_hat - coef)).item(),
            np.nan,
            np.nan,
            np.nan,
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished CoCo")


        # IRM
        t1 = time.time()
        coef_hat = irm_linear(
            X_list, y_list,
            penalty_weight=10,
            penalty_anneal_iters=100,
            l2_weight=1e-3,
            lr=1e-3,
            steps=500
        )
        t2 = time.time()
        rs = np.array([
            p,l,
            "IRM",
            (np.linalg.norm(coef_hat - coef)).item(),
            np.nan,
            np.nan,
            np.nan,
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished IRM")
    
        print(RS[(RS.shape[0]-8):RS.shape[0],:])
        print("finished p=", p, "l=", l, '\n')


finished MIP
finished EILLS
finished ICP
finished Dantzig
finished BIP
finished NegDRO
finished CoCo
finished IRM
[['100' '0' 'MIP-cd' '0.037202423964973044' 'True' 'True' '26'
  '1.7237548828125']
 ['100' '0' 'EILLS' '2.5495097567963922' 'False' 'False' '0'
  '5.096926927566528']
 ['100' '0' 'ICP' 'nan' 'False' 'False' '0' '6.868453025817871']
 ['100' '0' 'Causal Dantzig' '2.0509099646816917' 'False' 'False' '20'
  '0.3489952087402344']
 ['100' '0' 'BIP-s' '2.5495097567963922' 'False' 'False' '0'
  '0.3381631374359131']
 ['100' '0' 'NegDRO' '4.029758847724692' 'nan' 'nan' 'nan'
  '4.517486095428467']
 ['100' '0' 'CoCo' '0.10930190055538956' 'nan' 'nan' 'nan'
  '20.36532998085022']
 ['100' '0' 'IRM' '2.4313783711938983' 'nan' 'nan' 'nan'
  '1.1751952171325684']]
finished p= 100 l= 0 

finished MIP
finished EILLS
finished ICP
finished Dantzig
finished BIP
finished NegDRO
finished CoCo
finished IRM
[['100' '1' 'MIP-cd' '0.028013651327191046' 'True' 'True' '26'
  '1.682124137878418']
 ['1

#### E and $n_e$

In [ ]:
p=20
E_train=2

true_invariant, coef, X_list, y_list, n_list = generate_synthetic_data_soft_intervention(
    n=500,
    p=p,
    E=E_train,
    p_act=np.random.choice([0.1, 0.2]),
    lb=0.1, ub=0.1,
    lb_star=0.5, ub_star=1,
    sigma_x_min=0.5,
    sigma_x_max=1.0,
    sigma_y=0.5,
    p_star_min=4,
    p_star_max=5,
    p_resample = 0.1
)

true_invariant

array([ 2,  3,  7, 18, 19])

In [ ]:
from tqdm import tqdm

# RS = np.zeros((0, 9))
p = 20

n_grid = [50, 75, 100, 125, 150, 175, 200, 225, 250, 275]
E_grid = [2, 3, 5, 10]
n_trials = 100

pbar = tqdm(total=len(n_grid)*len(E_grid)*n_trials)

for n in n_grid:
    for E in E_grid:
        for l in range(n_trials):
            pbar.set_postfix(n=n, E=E, l=l)
            true_invariant, coef, X_list, y_list, n_list = generate_synthetic_data_soft_intervention(
                n=n,
                p=p,
                E=E,
                p_act=np.random.choice([0.1, 0.2]),
                lb=0.1, ub=0.1,
                lb_star=0.5, ub_star=1,
                sigma_x_min=0.5,
                sigma_x_max=1.0,
                sigma_y=0.5,
                p_star_min=4,
                p_star_max=5,
                p_resample = 0.05
            )

            # MIP-cd
            t1 = time.time()
            stat = precompute_stat(X_list, y_list, n_list=None)
            inv_set, B, obj = cd_msetype(
                stat, lam=0.01, gamma1=5, rho=1, forbid=None, S_init=(), superset_init=(), tol=1e-10, max_iter=100
            )
            t2 = time.time()
            coef_hat = B[0]
            rs = np.array([
                n, E, l,
                "CD",
                (np.linalg.norm(coef_hat - coef)).item(),
                set(inv_set).issuperset(set(true_invariant)),
                set(inv_set) == set(true_invariant),
                len(inv_set),
                t2 - t1
            ])
            RS = np.vstack([RS, rs])
            # print("finished CD")


            # vanilla MIP
            t1 = time.time()
            obj, B, g_sol = vanilla_msetype(
                X_list, y_list, n_list=None, gamma1=5, rho=1.0, lam=0.01,
                s_max=None, beta_bound=100.0, S_start=inv_set, B_start=None,
                mip_focus=2, mip_cuts=2, forbidden=None, time_limit=None,
                mip_gap=None, verbose=False
            )
            t2 = time.time()
            coef_hat = B[0]
            inv_set = np.where(g_sol!=0)[0]
            rs = np.array([
                n, E, l,
                "MIP",
                (np.linalg.norm(coef_hat - coef)).item(),
                set(inv_set).issuperset(set(true_invariant)),
                set(inv_set) == set(true_invariant),
                len(inv_set),
                t2 - t1
            ])
            RS = np.vstack([RS, rs])
            # print("finished IRM")
        
            # print(RS[(RS.shape[0]-2):RS.shape[0],:])
            # print("finished n=", n, "e=", E, "l=", l, '\n')
            pbar.update(1)

pbar.close()
            

100%|██████████| 2000/2000 [14:59<00:00,  2.22it/s, E=10, l=99, n=275]


In [ ]:
# pd.DataFrame(RS).to_csv("./summary_results/E_and_n/RS.csv", index=False, header=False)

#### Regularization

In [26]:
from tqdm import tqdm
RS = np.zeros((0, 10))

gamma_grid = [1, 5, 10, 15, 20]
lam_grid = [0.0001, 0.001, 0.01, 0.1, 1]
N_trial = 50

pbar = tqdm(total = len(gamma_grid) * len(lam_grid) * N_trial)

for gamma in gamma_grid:
    for lam in lam_grid:
        for l in range(N_trial):
            pbar.set_postfix(gamma=gamma, lam=lam, l=l)
            true_invariant, coef, X_list, y_list, n_list = generate_synthetic_data_soft_intervention(
                n=150,
                p=20,
                E=10,
                p_act=np.random.choice([0.1, 0.2]),
                lb=0.1, ub=0.1,
                lb_star=0.5, ub_star=1,
                sigma_x_min=0.5,
                sigma_x_max=1.0,
                sigma_y=0.5,
                p_star_min=4,
                p_star_max=5,
                p_resample = 0.05
            )

            # vanilla MIP
            loeo_val = loeo_mse(X_list, y_list, gamma1=gamma, rho=1, lam=lam, s_max=None).item()

            # obj, B, g_sol = vanilla_msetype(
            #     X_list, y_list, n_list=None, gamma1=gamma, rho=1.0, lam=lam,
            #     s_max=None, beta_bound=100.0, S_start=(), B_start=None,
            #     mip_focus=2, mip_cuts=2, forbidden=None, time_limit=None,
            #     mip_gap=None, verbose=False
            # )
            t1 = time.time()
            stat = precompute_stat(X_list, y_list, n_list=None)
            inv_set, B, obj = cd_msetype(
                stat, lam=lam, gamma1=gamma, rho=1, forbid=None, S_init=(), superset_init=(), tol=1e-10, max_iter=100
            )
            t2 = time.time()
            coef_hat = B[0]
            rs = np.array([
                gamma, lam, l,
                "MIP",
                (np.linalg.norm(coef_hat - coef)).item(),
                loeo_val,
                set(inv_set).issuperset(set(true_invariant)),
                set(inv_set) == set(true_invariant),
                len(inv_set),
                t2 - t1
            ])
            RS = np.vstack([RS, rs])

            pbar.update(1)
        
pbar.close()

100%|██████████| 1250/1250 [06:13<00:00,  3.35it/s, gamma=20, l=49, lam=1]     


#### Identification

In [ ]:
from tqdm import tqdm
RS = np.zeros([0, 7])

n_trail = 5000
pbar = tqdm(total = n_trail)

for l in range(n_trail):
    pbar.set_postfix(l=l)
    X_list, y_list, n_list = generate_X1_to_Y_to_X2_example(
        n_list=(500, 500),
        k_independent=8,
        gamma=1.0,
        tau2=1.0,
        sigma_eps2=1.0,
        a_list=(1.0, 0.5),
        v_list=(2.0, 1.5),
        seed=None,
        return_params=False,
    )
    E = 2
    p = X_list[0].shape[1]
    true_invariant = [0]
    coef = np.zeros(p)
    coef[true_invariant]=1



    ## MIP
    t1 = time.time()
    objval, B, g_sol = vanilla_msetype(X_list, y_list, n_list, gamma1=50, rho=1, lam=0.5, s_max=10, mip_focus=3, mip_cuts=3, S_start=None, time_limit = 60, verbose=0)
    t2 = time.time()
    inv_set = np.where(g_sol!=0)[0]
    coef_hat = B[0]
    rs = np.array([
        l,
        "MIP",
        (np.linalg.norm(coef_hat - coef)**2).item(),
        set(inv_set).issuperset(set(true_invariant)),
        set(inv_set) == set(true_invariant),
        set([1]).issubset(set(inv_set)),
        t2 - t1
    ])
    RS = np.vstack([RS, rs])



    ## EILLS
    t1 = time.time()
    what, g_eills = fit_eills(X_list, y_list, include_intercept=False, gamma=50)
    t2 = time.time()
    if len(g_eills) == 0:
        inv_set = []
    else:
        inv_set = g_eills 
    coef_hat = what
    rs = np.array([
        l,
        "EILLS",
        (np.linalg.norm(coef_hat - coef)**2).item(),
        set(inv_set).issuperset(set(true_invariant)),
        set(inv_set) == set(true_invariant),
        set([1]).issubset(set(inv_set)),
        t2 - t1
    ])
    RS = np.vstack([RS, rs])



    ## ICP
    data = []
    for i in range(len(X_list)):
        data.append( np.column_stack([X_list[i], y_list[i]]) )
    target = X_list[0].shape[1]
    t1 = time.time()
    result = icp.fit(data, target, alpha=0.05, precompute=True, verbose=False, color=False)
    t2 = time.time()
    if len(result.accepted_sets) == 0:
        intersection = []
    else:
        intersection = reduce(lambda a, b: a & b, result.accepted_sets)
    inv_set = list(intersection)
    rs = np.array([
        l,
        "ICP",
        np.nan,
        set(inv_set).issuperset(set(true_invariant)),
        set(inv_set) == set(true_invariant),
        set([1]).issubset(set(inv_set)),
        t2 - t1
    ])
    RS = np.vstack([RS, rs])
        


    ## Hidden ICP
    X = np.vstack(X_list)      # shape (n_total, p)
    Y = np.concatenate(y_list) # shape (n_total,)
    ExpInd = []
    start = 0
    for i, Xi in enumerate(X_list):
        n_i = Xi.shape[0]
        ExpInd.extend([i]*n_i)
    ExpInd = np.array(ExpInd)  # length n_total
    t1 = time.time()
    hiddenICPobj = fit_hiddenICP(X, Y, ExpInd, alpha=0.1, intercept=False)
    t2 = time.time()
    inv_set = np.where( hiddenICPobj['maximinCoefficients'] != 0 )[0]
    coef_hat = hiddenICPobj['maximinCoefficients']
    rs = np.array([
        l,
        "Causal Dantzig",
        (np.linalg.norm(coef_hat - coef)**2).item(),
        set(inv_set).issuperset(set(true_invariant)),
        set(inv_set) == set(true_invariant),
        set([1]).issubset(set(inv_set)),
        t2 - t1
    ])
    RS = np.vstack([RS, rs])



    ## BIP-s
    xs = []
    ys = []
    for i in range(len(X_list)):
        xs.append(torch.tensor(X_list[i]))
        ys.append(torch.tensor(y_list[i]))
    t1 = time.time()
    ols_result, log_posterior, posterior = pi_analytical(
        zs=None,        # None = enumerate all subsets automatically
        xs=xs,          # list of tensors or (E, N, p) tensor
        ys=ys,          # list of tensors or (E, N) tensor
        p_max=10,     # optional: restrict to subsets of size <= p_max
        epsilon=1e-4,   # ridge jitter for OLS stability
        use_vectorize=True,         # True = faster batched computation
        include_intercept=False,     # add intercept column to X
        keep_intercept=False         # always include intercept in z
    )
    t2 = time.time()
    p_ = xs[0].shape[-1]
    zs = get_all_zs(p_, p_max=10)   # all candidate z's in the same order as posterior
    map_idx = posterior.argmax()
    map_z = zs[map_idx]  
    inv_set = np.where(map_z !=0 )[0]
    coef_hat = np.zeros(p)
    coef_hat = ols_result[5][map_idx]
    rs = np.array([
        l,
        "BIP",
        (np.linalg.norm(coef_hat - coef)**2).item(),
        set(inv_set).issuperset(set(true_invariant)),
        set(inv_set) == set(true_invariant),
        set([1]).issubset(set(inv_set)),
        t2 - t1
    ])
    RS = np.vstack([RS, rs])

    pbar.update(1)

pbar.close()



  0%|          | 0/5000 [00:00<?, ?it/s, l=0]/var/folders/kz/bb7j3c312vq_tkp554m5_mgh0000gn/T/ipykernel_35012/396805969.py:151: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  (np.linalg.norm(coef_hat - coef)**2).item(),
100%|██████████| 5000/5000 [2:12:38<00:00,  1.59s/it, l=4999]  


In [ ]:
# pd.DataFrame(RS).to_csv("./summary_results/identification/RS1.csv", index=False, header=False)

#### Teaser/ example

In [661]:
S_invariant = {
    frozenset([0,1]), frozenset([0,2]), frozenset([0,1,2]), frozenset([0,1,3]),
    frozenset([0,2,3]), frozenset([0,1,2,3])
}

def is_invariant(S):
    return frozenset(S) in S_invariant

In [ ]:
RS = np.zeros([0, 9])

for n_ in [400, 800, 1200, 1600, 2000, 2400, 4000, 5000, 6000, 7000, 8000, 9000, 10000]:
    for l in range(200):

        X_train_list, y_train_list, n_train_list, X_test_list, y_test_list, n_test_list, coef = generate_multiple_invariant(
            n_train=300, n_test=n_,
            k=9,
            noise_std=1,
            lbd=1,
            ubd=1,
            perturb_strength=3
        )

        p = X_train_list[0].shape[1]
        true_invariant = [0,2]
        E = len(X_train_list)
        X_list = [np.vstack([X_train_list[e], X_test_list[e]]) for e in range(E)]
        y_list = [np.concatenate([y_train_list[e], y_test_list[e]]) for e in range(E)]
        n_list = [n_train_list[e] + n_test_list[e] for e in range(E)]

        X = np.vstack(X_list)          # shape: (sum(n_list), p)
        y = np.concatenate(y_list)     # shape: (sum(n_list),)
        n_es = n_list                  # e.g. [500, 500, 500]
        n = sum(n_es)
        superset, _ = ridge_screening(X_train_list, y_train_list, 15)
        super_structure_10, _ = ridge_screening(X_train_list, y_train_list, 10)

        
        ## MIP-2
        t1 = time.time()
        inv_set = mip_test_multiple(
            X_train_list, y_train_list, n_train_list, X_test_list, y_test_list, n_test_list, 
            superset, gamma=(1,1,0,0), alpha=.05,
            M_bar=5, B=300, solver="cd", verbose=0,
        )
        t2 = time.time()
        rs = np.array([
            n_, l, "MIP-test-2", 
            len(inv_set),
            set(inv_set) == set([0]),
            set(inv_set).issubset(set(true_invariant)),
            is_invariant(inv_set),
            len(set(inv_set) - {0,2}),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished MIP-2")


        ## MIP-4
        t1 = time.time()
        inv_set = mip_test_multiple(
            X_train_list, y_train_list, n_train_list, X_test_list, y_test_list, n_test_list, 
            superset, gamma=(1,1,1,1), alpha=.05,
            M_bar=5, B=300, solver="cd", verbose=0,
        )
        t2 = time.time()
        rs = np.array([
            n_, l, "MIP-test-4", 
            len(inv_set),
            set(inv_set) == set([0]),
            set(inv_set).issubset(set(true_invariant)),
            is_invariant(inv_set),
            len(set(inv_set) - {0,2}),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished MIP-4")


        ### ICP
        data = []
        for i in range(len(X_list)):
            data.append( np.column_stack([X_list[i], y_list[i]]) )
        target = X_list[0].shape[1]
        t1 = time.time()
        result = icp.fit(data, target, alpha=0.05, precompute=True, verbose=False, color=False)
        t2 = time.time()
        if len(result.accepted_sets) == 0:
            intersection = []
        else:
            intersection = reduce(lambda a, b: a & b, result.accepted_sets)
        inv_set = list(intersection)
        rs = np.array([
            n_, l, "ICP", 
            len(inv_set),
            set(inv_set) == set([0]),
            set(inv_set).issubset(set(true_invariant)),
            is_invariant(inv_set),
            len(set(inv_set) - {0,2}),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished ICP")


        ### Wasserstein
        wv = DirectWassersteinVariance(X, y, n_es, np.array(n_es) / n)
        _, causes, _, _, _ = WV_screening(wv, alpha=0.05, B=300)
        inv_set = np.where(causes == True)[0]
        rs = np.array([
            n_, l, "Wasserstein", 
            len(inv_set),
            set(inv_set) == set([0]),
            set(inv_set).issubset(set(true_invariant)),
            is_invariant(inv_set),
            len(set(inv_set) - {0,2}),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished Wasserstein")


        ### MIP
        t1 = time.time()
        stat = precompute_stat(X_list, y_list, n_list=None)
        inv_set, B, obj = cd_msetype(
            stat, lam=0.01, gamma1=5, rho=1, forbid=None, S_init=(), superset_init=(), tol=1e-10, max_iter=100
        )
        t2 = time.time()
        rs = np.array([
            n_, l, "MIP", 
            len(inv_set),
            set(inv_set) == set([0]),
            set(inv_set).issubset(set(true_invariant)),
            is_invariant(inv_set),
            len(set(inv_set) - {0,2}),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished MIP")


        ### EILLS
        t1 = time.time()
        coef_eills, g_eills = fit_eills(X_list, y_list, include_intercept=False, gamma=20)
        t2 = time.time()
        if len(g_eills) == 0:
            inv_set = []
        else:
            inv_set = g_eills 
        rs = np.array([
            n_, l, "EILLS", 
            len(inv_set),
            set(inv_set) == set([0]),
            set(inv_set).issubset(set(true_invariant)),
            is_invariant(inv_set),
            len(set(inv_set) - {0,2}),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished EILLS")


        ### Causal Dantzig
        X = np.vstack(X_list)      # shape (n_total, p)
        Y = np.concatenate(y_list) # shape (n_total,)
        ExpInd = []
        start = 0
        for i, Xi in enumerate(X_list):
            n_i = Xi.shape[0]
            ExpInd.extend([i]*n_i)
        ExpInd = np.array(ExpInd)  # length n_total
        t1 = time.time()
        hiddenICPobj = fit_hiddenICP(X, Y, ExpInd, alpha=0.05, intercept=False)
        t2 = time.time()
        inv_set = np.where( hiddenICPobj['maximinCoefficients'] != 0 )[0]
        rs = np.array([
            n_, l, "Causal Dantzig", 
            len(inv_set),
            set(inv_set) == set([0]),
            set(inv_set).issubset(set(true_invariant)),
            is_invariant(inv_set),
            len(set(inv_set) - {0,2}),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished Causal Dantzig")


        ### BIP-s
        X_list_screen = []
        for e in range(len(X_list)):
            X_list_screen.append( X_list[e][:,super_structure_10] )
        xs = []
        ys = []
        for i in range(len(X_list_screen)):
            xs.append(torch.tensor(X_list_screen[i]))
            ys.append(torch.tensor(y_list[i]))
        t1 = time.time()
        ols_result, log_posterior, posterior = pi_analytical(
            zs=None,        # None = enumerate all subsets automatically
            xs=xs,          # list of tensors or (E, N, p) tensor
            ys=ys,          # list of tensors or (E, N) tensor
            p_max=20,     # optional: restrict to subsets of size <= p_max
            epsilon=1e-4,   # ridge jitter for OLS stability
            use_vectorize=True,         # True = faster batched computation
            include_intercept=False,     # add intercept column to X
            keep_intercept=False         # always include intercept in z
        )
        t2 = time.time()
        p_ = xs[0].shape[-1]
        zs = get_all_zs(p_, p_max=10)   # all candidate z's in the same order as posterior
        map_idx = posterior.argmax()
        map_z = zs[map_idx]  
        inv_set = super_structure_10[np.where(map_z !=0 )[0]]
        rs = np.array([
            n_, l, "BIP-s", 
            len(inv_set),
            set(inv_set) == set([0]),
            set(inv_set).issubset(set(true_invariant)),
            is_invariant(inv_set),
            len(set(inv_set) - {0,2}),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished BIP-s")


        print("finished n=", n_, "l=", l)
        print(RS)



#### Vanilla vs CD

In [ ]:
from tqdm import tqdm

RS = np.zeros((0, 8))
p_grid = [10, 20, 30, 40, 50, 60]
n_trials = 100

pbar = tqdm(total=len(p_grid)*n_trials)

for p in p_grid:
    for l in range(n_trials):
        pbar.set_postfix(p=p, l=l)
        true_invariant, coef, X_list, y_list, n_list = generate_synthetic_data_soft_intervention(
            n=200,
            p=p,
            E=3,
            p_act=np.random.choice([0.1, 0.2]),
            lb=0.1, ub=0.1,
            lb_star=0.5, ub_star=1,
            sigma_x_min=0.5,
            sigma_x_max=1.0,
            sigma_y=0.5,
            p_star_min=4,
            p_star_max=5,
            p_resample = 0.05
        )

        # MIP-cd
        t1 = time.time()
        stat = precompute_stat(X_list, y_list, n_list=None)
        inv_set, B, obj = cd_msetype(
            stat, lam=0.01, gamma1=5, rho=1, forbid=None, S_init=(), superset_init=(), tol=1e-10, max_iter=100
        )
        t2 = time.time()
        coef_hat = B[0]
        rs = np.array([
            p, l,
            "CD",
            (np.linalg.norm(coef_hat - coef)).item(),
            set(inv_set).issuperset(set(true_invariant)),
            set(inv_set) == set(true_invariant),
            len(inv_set),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        # print("finished CD")


        # vanilla MIP
        t1 = time.time()
        obj, B, g_sol = vanilla_msetype(
            X_list, y_list, n_list=None, gamma1=5, rho=1.0, lam=0.01,
            s_max=None, beta_bound=100.0, S_start=inv_set, B_start=None,
            mip_focus=2, mip_cuts=2, forbidden=None, time_limit=180,
            mip_gap=None, verbose=False
        )
        t2 = time.time()
        coef_hat = B[0]
        inv_set = np.where(g_sol!=0)[0]
        rs = np.array([
            p, l,
            "MIP(warmStart)",
            (np.linalg.norm(coef_hat - coef)).item(),
            set(inv_set).issuperset(set(true_invariant)),
            set(inv_set) == set(true_invariant),
            len(inv_set),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])


        # vanilla MIP 2
        t1 = time.time()
        obj, B, g_sol = vanilla_msetype(
            X_list, y_list, n_list=None, gamma1=5, rho=1.0, lam=0.01,
            s_max=None, beta_bound=100.0, S_start=(), B_start=None,
            mip_focus=2, mip_cuts=2, forbidden=None, time_limit=180,
            mip_gap=None, verbose=False
        )
        t2 = time.time()
        coef_hat = B[0]
        inv_set = np.where(g_sol!=0)[0]
        rs = np.array([
            p, l,
            "MIP",
            (np.linalg.norm(coef_hat - coef)).item(),
            set(inv_set).issuperset(set(true_invariant)),
            set(inv_set) == set(true_invariant),
            len(inv_set),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
    
        # print(RS[(RS.shape[0]-2):RS.shape[0],:])
        # print("finished n=", n, "e=", E, "l=", l, '\n')
        pbar.update(1)

        print(RS[(RS.shape[0]-3):RS.shape[0],:])
        print('\n')

pbar.close()
            

In [702]:
# pd.DataFrame(RS).to_csv("./summary_results/vanilla_vs_cd/RS.csv", index=False, header=False)

#### Test-based

In [ ]:
p = 10
n = 500

RS = np.zeros((0, 7))

for E in range(3,13):
    for l in range(100):
        true_invariant, coef, X_list, y_list, n_list = generate_synthetic_data_all_intervention(
            n=n,
            p=p,
            E=E,
            p_act=np.random.choice([0.1,0.15]),
            lb=0.2, ub=0.5,
            lb_star=0.5, ub_star=0.5,
            sigma_x_min=0.5,
            sigma_x_max=1.0,
            sigma_y=0.5,
            p_star_min=4,
            p_star_max=5
        )

        X = np.vstack(X_list)          # shape: (sum(n_list), p)
        y = np.concatenate(y_list)     # shape: (sum(n_list),)
        n_es = n_list                  # e.g. [500, 500, 500]
        n = sum(n_es)
        
        X_train_list, y_train_list, n_train_list, X_test_list, y_test_list, n_test_list = train_test_split_envs(
            X_list, y_list, test_size=.8
        )


        ## MIP-2
        t1 = time.time()
        inv_set = mip_test_multiple(
            X_train_list, y_train_list, n_train_list, X_test_list, y_test_list, n_test_list, 
            superset=[j for j in range(p)], gamma=(1,1,0,0), alpha=.05,
            M_bar=3, B=500, solver="cd", verbose=0,
        )
        t2 = time.time()
        rs = np.array([
            n_, l, "MIP-test-2", 
            set(inv_set).issubset(set(true_invariant)),
            len(set(inv_set) & set(true_invariant)),
            set(inv_set) == set(true_invariant),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished MIP-2")

        
        ## MIP-4
        t1 = time.time()
        inv_set = mip_test_multiple(
            X_train_list, y_train_list, n_train_list, X_test_list, y_test_list, n_test_list, 
            superset=[j for j in range(p)], gamma=(1,1,1,1), alpha=.05,
            M_bar=3, B=500, solver="cd", verbose=0,
        )
        t2 = time.time()
        inv_set
        rs = np.array([
            n_, l, "MIP-test-4", 
            set(inv_set).issubset(set(true_invariant)),
            len(set(inv_set) & set(true_invariant)),
            set(inv_set) == set(true_invariant),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished MIP-4")

        
        ### ICP
        data = []
        for i in range(len(X_list)):
            data.append( np.column_stack([X_list[i], y_list[i]]) )
        target = X_list[0].shape[1]
        t1 = time.time()
        result = icp.fit(data, target, alpha=0.05, precompute=True, verbose=False, color=False)
        t2 = time.time()
        if len(result.accepted_sets) == 0:
            intersection = []
        else:
            intersection = reduce(lambda a, b: a & b, result.accepted_sets)
        inv_set = list(intersection)
        rs = np.array([
            n_, l, "ICP", 
            set(inv_set).issubset(set(true_invariant)),
            len(set(inv_set) & set(true_invariant)),
            set(inv_set) == set(true_invariant),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished ICP")


        ### Wasserstein
        t1 = time.time()
        wv = DirectWassersteinVariance(X, y, n_es, np.array(n_es) / n)
        _, causes, _, _, _ = WV_screening(wv, alpha=0.05, B=300)
        t2 = time.time()
        inv_set = np.where(causes == True)[0]
        rs = np.array([
            n_, l, "WVM", 
            set(inv_set).issubset(set(true_invariant)),
            len(set(inv_set) & set(true_invariant)),
            set(inv_set) == set(true_invariant),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished Wasserstein")
        
        print("finished E=", E, "l=", l)

[9, 7]

finished Wasserstein


array([1, 9])

#### hidden confounding

In [ ]:
from tqdm import tqdm

RS = np.zeros((0, 7))
n_grid = [1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000]
n_trials = 200

pbar = tqdm(total=len(n_grid)*n_trials)

for n in n_grid:
    for l in range(n_trials):

# for n in [5000]:
#     for l in range(1):
        pbar.set_postfix(n=n, l=l)

        E_train = 10
        p_a, p_d = np.random.choice([2,3], 2)
        X_list, y_list, n_list, A, true_invariant, D, I, coef, params = generate_hidden_sem_data(
            n=n, E=E_train,
            p_a=p_a, p_s=4, p_d=p_d, k=8, q=2,
            beta=np.ones(4),
            theta_D=np.full(p_d, .1),
            a_scales=[.2+.05*e for e in range(E_train)],
            s_scales=[.2+.05*e for e in range(E_train)],
            d_scales=[.2+.05*e for e in range(E_train)],
            sigma_y=.2,
            gamma_a_scale=.3,
            gamma_s_scale=.3,
            gamma_d_scale=.3,
            b_as_scale=.3
        )
        p = X_list[0].shape[1]
        super_structure, _ = ridge_screening(X_list, y_list, min(p, 100))
        super_structure_10, _ = ridge_screening(X_list, y_list, 10)
        super_structure_15, _ = ridge_screening(X_list, y_list, min(p, 15))



        ## MIP
        t1 = time.time()
        objval, B, g_sol = vanilla_msetype_additive(X_list, y_list, n_list, gamma=300, lam=0.5, s_max=10, mip_focus=3, mip_cuts=3, S_start=None, time_limit = 60, verbose = False)
        t2 = time.time()
        inv_set = np.where(g_sol!=0)[0]
        coef_hat = B
        rs = np.array([
            n,l,
            "MIP-hidden",
            (np.linalg.norm(coef_hat - coef)**2).item(),
            set(inv_set).issuperset(set(true_invariant)),
            set(inv_set) == set(true_invariant),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished MIP")



        ## EILLS
        X_list_screen = []
        for e in range(len(X_list)):
            X_list_screen.append( X_list[e][:,super_structure_15] )
        t1 = time.time()
        what, g_eills = fit_eills(X_list_screen, y_list, include_intercept=False, gamma=20)
        t2 = time.time()
        if len(g_eills) == 0:
            inv_set = []
        else:
            inv_set = super_structure_15[g_eills] 
        coef_hat = np.zeros(p)
        coef_hat[super_structure_15] = what
        rs = np.array([
            n,l,
            "EILLS-s",
            (np.linalg.norm(coef_hat - coef)**2).item(),
            set(inv_set).issuperset(set(true_invariant)),
            set(inv_set) == set(true_invariant),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished EILLS")



        # Hidden ICP
        X = np.vstack(X_list)      # shape (n_total, p)
        Y = np.concatenate(y_list) # shape (n_total,)
        ExpInd = []
        start = 0
        for i, Xi in enumerate(X_list):
            n_i = Xi.shape[0]
            ExpInd.extend([i]*n_i)
        ExpInd = np.array(ExpInd)  # length n_total
        t1 = time.time()
        hiddenICPobj = fit_hiddenICP(X, Y, ExpInd, alpha=0.1, intercept=False)
        t2 = time.time()
        inv_set = np.where( hiddenICPobj['maximinCoefficients'] != 0 )[0]
        coef_hat = hiddenICPobj['maximinCoefficients']
        rs = np.array([
            n,l,
            "Causal Dantzig",
            (np.linalg.norm(coef_hat - coef)**2).item(),
            set(inv_set).issuperset(set(true_invariant)),
            set(inv_set) == set(true_invariant),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished Causal Dantzig")



        # BIP-s
        X_list_screen = []
        for e in range(len(X_list)):
            X_list_screen.append( X_list[e][:,super_structure_10] )
        xs = []
        ys = []
        for i in range(len(X_list_screen)):
            xs.append(torch.tensor(X_list_screen[i]))
            ys.append(torch.tensor(y_list[i]))
        t1 = time.time()
        ols_result, log_posterior, posterior = pi_analytical(
            zs=None,        # None = enumerate all subsets automatically
            xs=xs,          # list of tensors or (E, N, p) tensor
            ys=ys,          # list of tensors or (E, N) tensor
            p_max=10,     # optional: restrict to subsets of size <= p_max
            epsilon=1e-4,   # ridge jitter for OLS stability
            use_vectorize=True,         # True = faster batched computation
            include_intercept=False,     # add intercept column to X
            keep_intercept=False         # always include intercept in z
        )
        t2 = time.time()
        p_ = xs[0].shape[-1]
        zs = get_all_zs(p_, p_max=10)   # all candidate z's in the same order as posterior
        map_idx = posterior.argmax()
        map_z = zs[map_idx]  
        inv_set = super_structure_10[np.where(map_z !=0 )[0]]
        coef_hat = np.zeros(p)
        coef_hat[super_structure_10] = ols_result[5][map_idx]
        rs = np.array([
            n,l,
            "BIP-s",
            (np.linalg.norm(coef_hat - coef)**2).item(),
            set(inv_set).issuperset(set(true_invariant)),
            set(inv_set) == set(true_invariant),
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished BIP")



        # negDRO
        t1 = time.time()
        b_final, _ = negDRO(X_list, y_list, gamma = 1000)
        t2 = time.time()
        coef_hat = b_final
        rs = np.array([
            n,l,
            "negDRO",
            (np.linalg.norm(coef_hat - coef)**2).item(),
            np.nan,
            np.nan,
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished negDRO")



        # CoCo
        t1 = time.time()
        res = fit_coco_linear_regularized(
            X_list=X_list,
            y_list=y_list,
            n_list=n_list,
            lam=10.0,
            lr=1e-2,
            max_iter=10000,
            include_intercept=False,
            equal_env_weight=True,
            penalty_sqrt=False,
            verbose=False,
        )
        t2 = time.time()
        coef_hat = res["beta"]
        rs = np.array([
            n,l,
            "CoCo",
            (np.linalg.norm(coef_hat - coef)**2).item(),
            np.nan,
            np.nan,
            t2 - t1
        ])
        RS = np.vstack([RS, rs])
        print("finished CoCo")

        pbar.update(1)


        print("finished", "n=",n, "l=",l, '\n')

pbar.close()

100%|██████████| 1800/1800 [06:40<00:00,  4.49it/s, l=199, n=5000]


In [ ]:
# pd.DataFrame(RS).to_csv("./summary_results/confounding/RS11.csv", index=False, header=False)

#### causal chamber

In [ ]:
from pathlib import Path

# Settings
DATA_DIR = Path("./causal_chamber_data/lt_interventions_standard_v1")
N_TRIALS, N_PER_ENV, BASE_SEED = 100, 1000, 2026
E_VALUES = range(3, 11)

# Variables
FEATURES = [
    "red", "green", "blue", "pol_1", "pol_2",
    "vis_1", "vis_2", "vis_3", "ir_1", "ir_2", "current"
]
TARGET = "ir_3"
TRUE_SUPPORT = ["red", "green", "blue", "pol_1", "pol_2"]

# Environments
REFERENCE_ENV = "uniform_reference"

INTERVENTION_POOL = [
    "uniform_red_strong",
    "uniform_green_strong",
    "uniform_blue_strong",
    "uniform_t_vis_1_strong",
    "uniform_t_vis_2_strong",
    "uniform_t_vis_3_strong",
    "uniform_t_ir_1_strong",
    "uniform_t_ir_2_strong",
    "uniform_diode_vis_1_mid",
    "uniform_diode_vis_2_mid",
    "uniform_diode_vis_3_mid",
    "uniform_diode_ir_1_strong",
    "uniform_diode_ir_2_strong",
    "uniform_l_11_mid",
    "uniform_l_12_mid",
    "uniform_l_21_mid",
    "uniform_l_22_mid",
    "uniform_osr_c_strong",
    "uniform_v_c_strong",
]

# Read the first 500 observations from each environment
def read_environment(name):
    df = pd.read_csv(DATA_DIR / f"{name}.csv").iloc[:N_PER_ENV].reset_index(drop=True)
    missing = [col for col in FEATURES + [TARGET] if col not in df.columns]
    if missing:
        raise ValueError(f"{name}.csv is missing columns: {missing}")
    return df

data = {REFERENCE_ENV: read_environment(REFERENCE_ENV)}
data.update({env: read_environment(env) for env in INTERVENTION_POOL})


# Construct nested environment sets
def construct_trials(n_trials=N_TRIALS, e_values=E_VALUES, base_seed=BASE_SEED):
    X_trials, y_trials, env_trials = {}, {}, {}
    # Standardization based only on the reference environment
    ref = data[REFERENCE_ENV]
    x_mean = ref[FEATURES].mean()
    x_std = ref[FEATURES].std(ddof=0).replace(0, 1.0)
    y_mean = ref[TARGET].mean()
    y_std = ref[TARGET].std(ddof=0)

    for trial in range(n_trials):
        rng = np.random.default_rng(base_seed + trial)
        order = list(rng.permutation(INTERVENTION_POOL))
        X_trials[trial], y_trials[trial], env_trials[trial] = {}, {}, {}
        sampled_data = {
            env: data[env].iloc[rng.choice(len(data[env]), N_PER_ENV, replace=True)]
            for env in [REFERENCE_ENV] + INTERVENTION_POOL
        }
        for E in e_values:
            envs = [REFERENCE_ENV] + order[:E - 1]
            X_list = [
                ((data[env][FEATURES] - x_mean) / x_std).to_numpy(dtype=float)
                for env in envs
            ]
            y_list = [
                ((data[env][TARGET] - y_mean) / y_std).to_numpy(dtype=float)
                for env in envs
            ]
            X_trials[trial][E] = X_list
            y_trials[trial][E] = y_list
            env_trials[trial][E] = envs

    return X_trials, y_trials, env_trials

X_trials, y_trials, env_trials = construct_trials()

In [1464]:
TRUE_SUPPORT_IDX = np.array([0, 1, 2, 3, 4])

def selection_metrics(selected, true_support=TRUE_SUPPORT_IDX):
    selected, true_support = set(selected), set(true_support)
    tp = len(selected & true_support)
    fp = len(selected - true_support)
    fn = len(true_support - selected)
    # precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    # recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    # f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    # fdr = fp / max(1, len(selected))
    return {"tp": tp, "fp": fp, "fn": fn, "size": len(selected)}

In [ ]:
# import pickle

# with open("causal_chamber_trials.pkl", "wb") as f:
#     pickle.dump({
#         "X_trials": X_trials,
#         "y_trials": y_trials,
#         "env_trials": env_trials
#     }, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
RS = np.zeros((0, 8))

for l in range(N_TRIALS):
    for e in E_VALUES:

        X_list = X_trials[l][e]
        y_list = y_trials[l][e]
        n_list = [x.shape[0] for x in X_list]

        # MIP-cd
        t1 = time.time()
        stat = precompute_stat(X_list, y_list, n_list=None)
        mip_inv_set, B, obj = cd_msetype(
            stat, lam=0.01, gamma1=5, rho=1, forbid=None, S_init=(), superset_init=(), tol=1e-10, max_iter=100
        )
        t2 = time.time()
        metrics = selection_metrics(mip_inv_set)
        rs = np.array([e, l, 'MIP', metrics['tp'], metrics['fp'], metrics['fn'], metrics['size'], t2-t1 ])
        RS = np.vstack([RS, rs])


        # EILLS
        t1 = time.time()
        what, g_eills = fit_eills(X_list, y_list, include_intercept=False, gamma=20)
        t2 = time.time()
        if len(g_eills) == 0:
            inv_set = []
        else:
            inv_set = g_eills
        eills_inv_set = inv_set
        metrics = selection_metrics(eills_inv_set)
        rs = np.array([e, l, 'EILLS', metrics['tp'], metrics['fp'], metrics['fn'], metrics['size'], t2-t1 ])
        RS = np.vstack([RS, rs])


        # ICP
        data = []
        for i in range(len(X_list)):
            data.append( np.column_stack([X_list[i], y_list[i]]) )
        target = X_list[0].shape[1]
        t1 = time.time()
        result = icp.fit(data, target, alpha=0.05, precompute=True, verbose=False, color=False)
        t2 = time.time()
        if len(result.accepted_sets) ==0:
            intersection=[]
        else:
            intersection = reduce(lambda a, b: a & b, result.accepted_sets)
        icp_inv_set = intersection
        metrics = selection_metrics(icp_inv_set)
        rs = np.array([e, l, 'ICP', metrics['tp'], metrics['fp'], metrics['fn'], metrics['size'], t2-t1 ])
        RS = np.vstack([RS, rs])


        # Causal Dantzig
        X = np.vstack(X_list)      # shape (n_total, p)
        Y = np.concatenate(y_list) # shape (n_total,)
        ExpInd = []
        for i, Xi in enumerate(X_list):
            n_i = Xi.shape[0]
            ExpInd.extend([i]*n_i)
        ExpInd = np.array(ExpInd)  # length n_total
        t1 = time.time()
        dantzig_inv_set = np.where( fit_hiddenICP(X, Y, ExpInd, alpha=0.05, intercept=False)['maximinCoefficients'] != 0 )[0]
        t2 = time.time()
        metrics = selection_metrics(dantzig_inv_set)
        rs = np.array([e, l, 'Causal Dantzig', metrics['tp'], metrics['fp'], metrics['fn'], metrics['size'], t2-t1 ])
        RS = np.vstack([RS, rs])

        
        # BIP
        xs = []
        ys = []
        for i in range(len(X_list)):
            xs.append(torch.tensor(X_list[i]))
            ys.append(torch.tensor(y_list[i]))
        t1 = time.time()
        ols_result, log_posterior, posterior = pi_analytical(
            zs=None,        # None = enumerate all subsets automatically
            xs=xs,          # list of tensors or (E, N, p) tensor
            ys=ys,          # list of tensors or (E, N) tensor
            p_max=10,     # optional: restrict to subsets of size <= p_max
            epsilon=1e-4,   # ridge jitter for OLS stability
            use_vectorize=True,         # True = faster batched computation
            include_intercept=False,     # add intercept column to X
            keep_intercept=False         # always include intercept in z
        )
        t2 = time.time()
        p_ = xs[0].shape[-1]
        zs = get_all_zs(p_, p_max=10)   # all candidate z's in the same order as posterior
        map_idx = posterior.argmax()
        map_z = zs[map_idx]  
        bip_inv_set = np.where(map_z !=0 )[0]
        metrics = selection_metrics(bip_inv_set)
        rs = np.array([e, l, 'BIP', metrics['tp'], metrics['fp'], metrics['fn'], metrics['size'], t2-t1 ])
        RS = np.vstack([RS, rs])


        # MIP-test
        t1 = time.time()
        X_mip_list, y_mip_list, n_mip_list, X_diag_list, y_diag_list, n_diag_list = train_test_split_envs(
            X_list, y_list, test_size=.8
        )
        stat = precompute_stat(X_mip_list, y_mip_list, n_list=None)
        mip_inv_set, B, obj = cd_msetype(
            stat, lam=0.01, gamma1=5, rho=1, forbid=None, S_init=(), superset_init=(), tol=1e-10, max_iter=100
        )
        S_test = mip_inv_set
        X_list_screen = []
        for e_ in range(len(X_diag_list)):
            X_e_screen = X_diag_list[e_][:, S_test]  # select features
            X_list_screen.append(X_e_screen)
        X_train_list, y_train_list, n_train_list, X_test_list, y_test_list, n_test_list = train_test_split_envs(
            X_list_screen, y_diag_list, test_size=.8
        )
        inv_set = mip_test_multiple(
            X_train_list, y_train_list, n_train_list, X_test_list, y_test_list, n_test_list, 
            superset=[j for j in range(X_train_list[0].shape[1])], gamma=(1,1,1,1), alpha=.05,
            M_bar=5, B=500, solver="cd", verbose=0,
        )
        t2 = time.time()
        MIPtest_inv_set = np.array(mip_inv_set)[inv_set]
        metrics = selection_metrics(MIPtest_inv_set)
        rs = np.array([e, l, 'MIPtest', metrics['tp'], metrics['fp'], metrics['fn'], metrics['size'], t2-t1 ])
        RS = np.vstack([RS, rs])


        # FAIR
        t1 = time.time()
        fair_inv_set, probs, gate_history = fit_fair_nn(X_list, y_list, gamma=36, niters=10000, threshold=.9)
        t2 = time.time()
        metrics = selection_metrics(fair_inv_set)
        rs = np.array([e, l, 'FAIR', metrics['tp'], metrics['fp'], metrics['fn'], metrics['size'], t2-t1 ])
        RS = np.vstack([RS, rs])


        print('finished', 'l=', l, 'e=', e)
        print(RS[(RS.shape[0]-6):RS.shape[0],:])

#### nonlinear

In [ ]:
RS = np.zeros((0, 6))

for l in range(100):
    for n_ in [100, 150, 200, 250, 300, 350, 400, 450, 500]:
        
        true_invariant, beta_star, X_list, y_list, n_list = generate_synthetic_data_all_intervention_nonlinear(
            n=n_,
            p=6,
            E=5,
            p_act=0.2,
            lb=0.1,
            ub=0.5,
            lb_star=0.5,
            ub_star=1.0,
            sigma_x_min=0.5,
            sigma_x_max=1.0,
            sigma_y=0.5,
            p_star_min=2,
            p_star_max=2
        )
        X = np.vstack(X_list)          # shape: (sum(n_list), p)
        y = np.concatenate(y_list)     # shape: (sum(n_list),)
        n_es = n_list                  # e.g. [500, 500, 500]
        n = sum(n_es)

        df = pd.DataFrame(X, columns=[f"X{j}" for j in range(X.shape[1])])
        df["Y"] = y
        df["E"] = np.repeat(np.arange(1, len(n_list)+1), n_list)
        df.to_csv("data" + str(n_) + "_" + str(l) + ".csv", index=False)


        ## MIP
        K = 2
        Z_list, groups = fourier_expand(X_list, K=K)
        t1 = time.time()
        stat = precompute_stat(Z_list, y_list, n_list)
        inv_set, B, obj = cd_msetype_grouped(
            stat, groups, lam=.01, gamma1=1/K , rho=1
        )
        t2 = time.time()
        rs = np.array([
            n_, l, 'MIP', 
            set(inv_set) == set(true_invariant),
            len(list(set(inv_set) & set(true_invariant))),
            t2 - t1,
        ])
        RS = np.vstack([RS, rs])
        print("finished MIP")


        ## Wasserstein
        t1 = time.time()
        wv = DirectWassersteinVariance(X, y, n_es, np.array(n_es) / n)
        _, causes, _, _, _ = WV_screening(wv, alpha=0.05, B=300)
        t2 = time.time()
        inv_set = np.where(causes == True)[0]
        rs = np.array([
            n_, l, "Wasserstein", 
            set(inv_set) == set(true_invariant),
            len(list(set(inv_set) & set(true_invariant))),
            t2 - t1,
        ])
        RS = np.vstack([RS, rs])
        print("finished Wasserstein")


        ## FAIR
        t1 = time.time()
        inv_set, probs, gate_history = fit_fair_nn(X_list, y_list, gamma=36, niters=10000, threshold=.9)
        t2 = time.time()
        rs = np.array([
            n_, l, "FAIR", 
            set(inv_set) == set(true_invariant),
            len(list(set(inv_set) & set(true_invariant))),
            t2 - t1,
        ])
        RS = np.vstack([RS, rs])
        print("finished FAIR")


        print('finished', 'l=', l, 'e=', e)
        print(RS[(RS.shape[0]-3):RS.shape[0],:])


In [2200]:
# true_invariant, beta_star, X_list, y_list, n_list = generate_synthetic_data_all_intervention_nonlinear(
#     n=n_,
#     p=6,
#     E=5,
#     p_act=0.2,
#     lb=0.1,
#     ub=0.5,
#     lb_star=0.5,
#     ub_star=1.0,
#     sigma_x_min=0.5,
#     sigma_x_max=1.0,
#     sigma_y=0.5,
#     p_star_min=2,
#     p_star_max=2
# )
# X = np.vstack(X_list)          # shape: (sum(n_list), p)
# y = np.concatenate(y_list)     # shape: (sum(n_list),)
# n_es = n_list                  # e.g. [500, 500, 500]
# n = sum(n_es)

## MIP
K = 3
Z_list, groups = fourier_expand(X_list, K=K)
t1 = time.time()
stat = precompute_stat(Z_list, y_list, n_list)
inv_set, B, obj = cd_msetype_grouped(
    stat, groups, lam=.00001, gamma1=1/K , rho=1
)
t2 = time.time()

true_invariant, inv_set


(array([1, 5]), [1])

In [931]:
# for j in S_star:
#     print("\nfeature", j)
#     for e, X in enumerate(X_list):
#         z = beta_star[j] * np.sin(X[:, j])
#         print(e, z.mean(), z.var())

In [932]:
# for e, X in enumerate(X_list):
#     signal = sum(beta_star[j] * np.sin(X[:, j]) for j in S_star)
#     print(e, signal.mean(), signal.var())

In [813]:
# from itertools import combinations

# def exhaustive_grouped(stat, groups, gamma1, rho, lam):
#     p = len(groups)
#     out = []

#     for s in range(p + 1):
#         for S in combinations(range(p), s):
#             B = compute_B_given_groups(S, stat, groups)
#             obj, parts = compute_mse_obj_given_groups(
#                 B, S, stat, gamma1, rho, lam, return_parts=True
#             )
#             out.append((list(S), obj, parts))

#     return sorted(out, key=lambda x: x[1])

# res = exhaustive_grouped(
#     stat, groups,
#     gamma1=1,
#     rho=1,
#     lam=0.001
# )

# for S, obj, parts in res[:10]:
#     print(S, obj, parts)

In [814]:
# sets = [
#     [],
#     [S_star[0]],
#     [S_star[1]],
#     list(S_star)
# ]

# for S in sets:
#     B = compute_B_given_groups(S, stat, groups)
#     obj, parts = compute_mse_obj_given_groups(
#         B, S, stat,
#         gamma1=1, rho=1, lam=0.001,
#         return_parts=True
#     )
#     print("\nS =", S)
#     print(parts)

#### next